# DocuMind RAG: An Explainable NLP and Retrieval-Augmented Question Answering System for Academic and Technical Documents

## NLP Course Project

This project builds an explainable question-answering system for academic and technical documents such as research papers, lecture notes, technical articles, reports, and manuals.

The main goal of this project is not only to build a final question-answering system, but also to demonstrate a complete Natural Language Processing pipeline before applying advanced models such as embeddings, large language models, and Retrieval-Augmented Generation.

The project first demonstrates classical NLP techniques including text extraction, cleaning, tokenization, normalization, stopword removal, stemming, lemmatization, POS tagging, dependency parsing, and encoding techniques such as Bag of Words, TF-IDF, and embeddings.

After completing the classical NLP pipeline, the project builds and compares three approaches:

1. A TF-IDF based baseline retrieval system
2. An embedding-based semantic retrieval system
3. A Retrieval-Augmented Generation based question-answering system

The final system is designed to be explainable by showing the document chunks used to retrieve and generate answers.

## 1. Problem Statement

Academic and technical documents often contain large amounts of complex information. Students, researchers, and professionals may find it difficult to quickly locate specific answers from long research papers, lecture notes, technical reports, or manuals.

Traditional keyword search can retrieve relevant sections, but it may fail when the question uses different wording from the document. Modern language models can generate fluent answers, but they may produce unsupported or hallucinated responses if they are not grounded in source documents.

Therefore, this project aims to build an explainable NLP-based question-answering system that retrieves relevant content from academic and technical documents and uses that content to answer user questions.

## 2. Project Objectives

The objectives of this project are:

1. To extract machine-readable text from academic and technical PDF documents.
2. To demonstrate a complete classical NLP preprocessing pipeline.
3. To apply tokenization, normalization, stopword removal, stemming, lemmatization, POS tagging, and dependency parsing.
4. To demonstrate encoding techniques including Bag of Words, TF-IDF, and sentence embeddings.
5. To build a classical TF-IDF based document retrieval system.
6. To build an embedding-based semantic retrieval system using sentence embeddings and FAISS.
7. To build a Retrieval-Augmented Generation based question-answering system.
8. To make the system explainable by displaying retrieved chunks and source context.
9. To compare classical retrieval, semantic retrieval, and RAG-based question answering.

## 3. Scope of the Project

This project focuses specifically on academic and technical documents, including:

- Research papers
- Lecture notes
- Technical articles
- Educational PDFs
- Reports
- Manuals

This project is not designed as a general-purpose question-answering system for every type of document. It is positioned as a question-answering system for structured and semi-structured academic or technical text.

Since this project focuses on Natural Language Processing, only machine-readable textual content will be extracted and processed. Images, diagrams, charts, and non-textual figures will be excluded from the pipeline. Captions will be retained only if they appear as extractable text.

OCR-based image text extraction is considered outside the scope of this project and can be added as future work.

In [99]:
# Install required Python libraries

!pip install pymupdf
!pip install nltk
!pip install spacy
!pip install scikit-learn
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers accelerate
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [100]:
# Common Python libraries
import os
import re
import warnings

# Data handling libraries
import numpy as np
import pandas as pd

# PDF text extraction
import fitz  # PyMuPDF

# NLP libraries
import nltk
import spacy

# Classical NLP feature extraction and similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Display utilities
from IPython.display import display

# Ignore unnecessary warnings
warnings.filterwarnings("ignore")

print("Common libraries imported successfully.")

Common libraries imported successfully.


In [101]:
# Download required NLTK resources

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("omw-1.4")

print("NLTK resources downloaded successfully.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


NLTK resources downloaded successfully.


[nltk_data]   Package omw-1.4 is already up-to-date!


In [102]:
# Load spaCy English language model

nlp = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully.")

spaCy model loaded successfully.


In [103]:
# Creating folder structure inside Colab

folders = [
    "data",
    "data/raw_documents",
    "data/processed_text",
    "outputs",
    "outputs/retrieved_chunks",
    "outputs/evaluation"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folder structure created successfully.")

Project folder structure created successfully.


In [104]:
# Display created folders

for root, dirs, files in os.walk("data"):
    level = root.replace("data", "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)

    for file in files:
        print(f"{subindent}{file}")

data/
    raw_documents/
        bert.pdf
        rag.pdf
        attention_is_all_you_need.pdf
    processed_text/
        attention_is_all_you_need.txt
        bert_normalized.txt
        rag_cleaned.txt
        rag.txt
        rag_stopword_removed.txt
        bert_lemmatized.txt
        attention_is_all_you_need_cleaned.txt
        attention_is_all_you_need_normalized.txt
        bert_stopword_removed.txt
        bert_cleaned.txt
        bert.txt
        attention_is_all_you_need_lemmatized.txt
        attention_is_all_you_need_stopword_removed.txt
        rag_normalized.txt
        rag_lemmatized.txt


## 4. Dataset / Document Collection

This project uses a small collection of academic and technical documents related to Natural Language Processing, Transformer models, and Retrieval-Augmented Generation.

The selected documents are research papers that are text-heavy, machine-readable, and directly relevant to the development of modern NLP systems. These documents are suitable for this project because they contain structured academic writing, technical terminology, explanations of models, methodology sections, experimental results, and references.

The document collection includes:

1. **Attention Is All You Need**  
   This paper introduces the Transformer architecture, which is one of the most important foundations of modern NLP and large language models.

2. **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding**  
   This paper introduces BERT, a bidirectional Transformer-based language representation model used for many NLP tasks.

3. **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks**  
   This paper introduces Retrieval-Augmented Generation, which combines retrieval-based methods with generative language models.

These documents were selected because they support the full learning path of this project: from classical NLP processing to embeddings, semantic retrieval, and RAG-based question answering.

## 4.1 Document Scope and Image Handling

Since this project focuses on Natural Language Processing, only machine-readable textual content was extracted and processed.

Images, diagrams, charts, tables, and non-textual figures were excluded from the NLP pipeline. Captions were retained only if they appeared as extractable text in the PDF.

OCR-based image text extraction is considered outside the scope of this project and can be added as future work.

This decision keeps the project focused on text preprocessing, representation, retrieval, and question answering.

In [105]:
# Define selected academic and technical documents

documents = {
    "attention_is_all_you_need.pdf": {
        "title": "Attention Is All You Need",
        "topic": "Transformer architecture and self-attention",
        "url": "https://arxiv.org/pdf/1706.03762",
        "reason": "Introduces the Transformer architecture, a foundation of modern NLP and large language models."
    },
    "bert.pdf": {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "topic": "Bidirectional Transformer-based language representation",
        "url": "https://arxiv.org/pdf/1810.04805",
        "reason": "Explains BERT, a major Transformer-based NLP model used for language understanding tasks."
    },
    "rag.pdf": {
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "topic": "Retrieval-Augmented Generation",
        "url": "https://arxiv.org/pdf/2005.11401",
        "reason": "Directly supports the RAG component of this project by explaining retrieval-augmented generation."
    }
}

print("Document sources defined successfully.")

Document sources defined successfully.


In [106]:
# Download the selected PDFs into the raw_documents folder

for filename, info in documents.items():
    output_path = f"data/raw_documents/{filename}"
    url = info["url"]

    if not os.path.exists(output_path):
        print(f"Downloading: {filename}")
        !wget -O "{output_path}" "{url}"
    else:
        print(f"{filename} already exists.")

print("Document download process completed.")

attention_is_all_you_need.pdf already exists.
bert.pdf already exists.
rag.pdf already exists.
Document download process completed.


In [107]:
# Create metadata table for selected documents

document_metadata = []

for filename, info in documents.items():
    file_path = f"data/raw_documents/{filename}"

    document_metadata.append({
        "File Name": filename,
        "Title": info["title"],
        "Topic": info["topic"],
        "Reason for Selection": info["reason"],
        "File Path": file_path
    })

document_metadata_df = pd.DataFrame(document_metadata)

display(document_metadata_df)

,File Name,Title,Topic,Reason for Selection,File Path
0,attention_is_all_you_need.pdf,Attention Is All You Need,Transformer architecture and self-attention,"Introduces the Transformer architecture, a fou...",data/raw_documents/attention_is_all_you_need.pdf
1,bert.pdf,BERT: Pre-training of Deep Bidirectional Trans...,Bidirectional Transformer-based language repre...,"Explains BERT, a major Transformer-based NLP m...",data/raw_documents/bert.pdf
2,rag.pdf,Retrieval-Augmented Generation for Knowledge-I...,Retrieval-Augmented Generation,Directly supports the RAG component of this pr...,data/raw_documents/rag.pdf


In [108]:
# Verify that the PDF files were downloaded successfully

print("Files in data/raw_documents:")

for file in os.listdir("data/raw_documents"):
    file_path = os.path.join("data/raw_documents", file)
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"- {file} ({file_size_mb:.2f} MB)")

Files in data/raw_documents:
- bert.pdf (0.74 MB)
- rag.pdf (0.84 MB)
- attention_is_all_you_need.pdf (2.11 MB)


## 5. Text Extraction

In this phase, text is extracted from the selected PDF documents using PyMuPDF.

The purpose of text extraction is to convert academic PDF documents into plain text so that NLP preprocessing techniques can be applied.

Only machine-readable text is extracted. Images, diagrams, charts, and scanned content are not processed in this project. Captions are retained only if they are available as extractable text inside the PDF.

In [109]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts machine-readable text from a PDF file using PyMuPDF.

    Parameters:
    pdf_path (str): Path to the PDF file.

    Returns:
    tuple:
        full_text (str): Extracted text from the entire PDF.
        page_count (int): Number of pages in the PDF.
    """

    full_text = ""

    with fitz.open(pdf_path) as pdf:
        page_count = len(pdf)

        for page_number, page in enumerate(pdf, start=1):
            page_text = page.get_text()

            # Add page marker to understand where text came from
            full_text += f"\n\n--- Page {page_number} ---\n\n"
            full_text += page_text

    return full_text, page_count

In [110]:
# Extract text from all selected PDF documents

raw_texts = {}
extraction_summary = []

for filename in documents.keys():
    pdf_path = f"data/raw_documents/{filename}"

    extracted_text, page_count = extract_text_from_pdf(pdf_path)

    raw_texts[filename] = extracted_text

    extraction_summary.append({
        "File Name": filename,
        "Number of Pages": page_count,
        "Extracted Characters": len(extracted_text),
        "Extracted Words Approx.": len(extracted_text.split())
    })

print("Text extraction completed for all documents.")

Text extraction completed for all documents.


In [111]:
# Save extracted text into processed_text folder

for filename, text in raw_texts.items():
    txt_filename = filename.replace(".pdf", ".txt")
    txt_path = f"data/processed_text/{txt_filename}"

    with open(txt_path, "w", encoding="utf-8") as file:
        file.write(text)

print("Extracted text files saved successfully.")

Extracted text files saved successfully.


In [112]:
# Display extraction summary as a table

extraction_summary_df = pd.DataFrame(extraction_summary)

display(extraction_summary_df)

,File Name,Number of Pages,Extracted Characters,Extracted Words Approx.
0,attention_is_all_you_need.pdf,15,39774,6155
1,bert.pdf,16,64412,10216
2,rag.pdf,19,69430,9962


In [113]:
# Verify saved extracted text files

print("Files in data/processed_text:")

for file in os.listdir("data/processed_text"):
    file_path = os.path.join("data/processed_text", file)
    file_size_kb = os.path.getsize(file_path) / 1024
    print(f"- {file} ({file_size_kb:.2f} KB)")

Files in data/processed_text:
- attention_is_all_you_need.txt (38.99 KB)
- bert_normalized.txt (59.19 KB)
- rag_cleaned.txt (64.99 KB)
- rag.txt (68.39 KB)
- rag_stopword_removed.txt (51.09 KB)
- bert_lemmatized.txt (45.89 KB)
- attention_is_all_you_need_cleaned.txt (28.79 KB)
- attention_is_all_you_need_normalized.txt (27.76 KB)
- bert_stopword_removed.txt (47.64 KB)
- bert_cleaned.txt (62.54 KB)
- bert.txt (63.49 KB)
- attention_is_all_you_need_lemmatized.txt (20.82 KB)
- attention_is_all_you_need_stopword_removed.txt (21.78 KB)
- rag_normalized.txt (61.87 KB)
- rag_lemmatized.txt (49.37 KB)


In [114]:
# Display raw extracted text sample from one document

sample_document = "attention_is_all_you_need.pdf"

print("Raw extracted text sample from:", sample_document)
print("=" * 80)
print(raw_texts[sample_document][:2000])

Raw extracted text sample from: attention_is_all_you_need.pdf


--- Page 1 ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on a

## 5.1 Text Extraction Limitations

The extracted text may contain formatting noise because PDF files are designed mainly for visual presentation rather than direct text processing.

Common extraction issues include:

- Broken lines
- Extra spaces
- Page numbers
- Headers and footers
- Reference section noise
- Symbols from equations
- Text appearing in an unusual order because of PDF layout

Since this project focuses on NLP, only machine-readable text is used. Images, diagrams, and scanned content are excluded. These limitations will be partly addressed in the text cleaning stage.

## 6. Text Cleaning

Text cleaning is an important preprocessing step in Natural Language Processing.

The text extracted from PDF documents often contains noise such as page markers, unnecessary line breaks, extra spaces, page numbers, headers, footers, and formatting issues. If this noise is not handled, it can negatively affect tokenization, feature extraction, retrieval, and question answering.

In this project, text cleaning is applied to improve the quality of the extracted academic and technical document text before further NLP processing.

In [115]:
# Text cleaning function
def clean_text(text):
    """
    Cleans raw extracted PDF text from academic and technical documents.

    Cleaning steps:
    1. Remove artificial page markers added during extraction.
    2. Remove URLs and email addresses.
    3. Remove common academic PDF boilerplate text.
    4. Remove front-matter content before the Abstract section when Abstract appears early.
    5. Remove reference or bibliography sections near the end.
    6. Remove excessive newline characters, tabs, and spaces.
    7. Standardise spacing.

    Parameters:
    text (str): Raw extracted text.

    Returns:
    str: Cleaned text.
    """

    # Remove page markers such as --- Page 1 ---
    text = re.sub(r"--- Page \d+ ---", " ", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove common arXiv / permission boilerplate if present
    text = re.sub(
        r"Provided proper attribution is provided,.*?scholarly works\.",
        " ",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    # Remove standalone page numbers
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    # Remove excessive newline characters
    text = re.sub(r"\n+", " ", text)

    # Remove tabs
    text = re.sub(r"\t+", " ", text)

    # Remove front matter before Abstract if Abstract appears early in the document
    abstract_match = re.search(r"\bAbstract\b", text, flags=re.IGNORECASE)

    if abstract_match and abstract_match.start() < 5000:
        text = text[abstract_match.start():]

    # Remove references or bibliography section if it appears in the later part of the document
    reference_patterns = [
        r"\bReferences\b",
        r"\bBibliography\b"
    ]

    cut_positions = []

    for pattern in reference_patterns:
        matches = list(re.finditer(pattern, text, flags=re.IGNORECASE))

        for match in matches:
            if match.start() > len(text) * 0.5:
                cut_positions.append(match.start())

    if cut_positions:
        text = text[:min(cut_positions)]

    # Remove excessive spaces
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text

In [116]:
# Apply text cleaning to all extracted documents

cleaned_texts = {}

for filename, text in raw_texts.items():
    cleaned_texts[filename] = clean_text(text)

print("Text cleaning completed for all documents.")

Text cleaning completed for all documents.


In [117]:
# Compare raw and cleaned text from one sample document

sample_document = "attention_is_all_you_need.pdf"

print("RAW TEXT SAMPLE")
print("=" * 80)
print(raw_texts[sample_document][:2000])

print("\n\nCLEANED TEXT SAMPLE")
print("=" * 80)
print(cleaned_texts[sample_document][:2000])

RAW TEXT SAMPLE


--- Page 1 ---

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrenc

In [118]:
# Create summary table showing text size before and after cleaning

cleaning_summary = []

for filename in raw_texts.keys():
    raw_characters = len(raw_texts[filename])
    cleaned_characters = len(cleaned_texts[filename])

    raw_words = len(raw_texts[filename].split())
    cleaned_words = len(cleaned_texts[filename].split())

    cleaning_summary.append({
        "File Name": filename,
        "Raw Characters": raw_characters,
        "Cleaned Characters": cleaned_characters,
        "Raw Words Approx.": raw_words,
        "Cleaned Words Approx.": cleaned_words,
        "Characters Removed": raw_characters - cleaned_characters
    })

cleaning_summary_df = pd.DataFrame(cleaning_summary)

display(cleaning_summary_df)

,File Name,Raw Characters,Cleaned Characters,Raw Words Approx.,Cleaned Words Approx.,Characters Removed
0,attention_is_all_you_need.pdf,39774,29395,6155,4589,10379
1,bert.pdf,64412,37140,10216,6029,27272
2,rag.pdf,69430,36788,9962,5681,32642


The initial extracted text contained front-page academic metadata such as author names, affiliations, email addresses, permission statements, and formatting artefacts. These elements are not useful for the main goal of this project, which is question answering over academic and technical content.

Therefore, the cleaning function removes emails, URLs, permission boilerplate, and front matter before the Abstract section when appropriate. This keeps the focus on the actual technical content of the documents.

The reference section is also removed when it appears near the end of the document because references can introduce many unrelated names, titles, and citations that may add noise to retrieval and question answering.

The reference section was removed from the cleaned text used for retrieval and question answering. This decision was made because references often contain many author names, paper titles, publication venues, years, and citation-related text that may introduce noise into Bag of Words, TF-IDF, embedding retrieval, and RAG-based answer generation.

Since the goal of this project is to answer questions based on the main academic and technical content of the documents, sections such as the abstract, introduction, methodology, experiments, results, and conclusion are more relevant than the bibliography. The original extracted text is still preserved separately, so the cleaning process remains transparent.

In [119]:
# Save cleaned text files

for filename, text in cleaned_texts.items():
    cleaned_filename = filename.replace(".pdf", "_cleaned.txt")
    cleaned_path = f"data/processed_text/{cleaned_filename}"

    with open(cleaned_path, "w", encoding="utf-8") as file:
        file.write(text)

print("Cleaned text files saved successfully.")

Cleaned text files saved successfully.


In [120]:
# Verify saved cleaned text files

print("Files in data/processed_text:")

for file in os.listdir("data/processed_text"):
    file_path = os.path.join("data/processed_text", file)
    file_size_kb = os.path.getsize(file_path) / 1024
    print(f"- {file} ({file_size_kb:.2f} KB)")

Files in data/processed_text:
- attention_is_all_you_need.txt (38.99 KB)
- bert_normalized.txt (59.19 KB)
- rag_cleaned.txt (36.30 KB)
- rag.txt (68.39 KB)
- rag_stopword_removed.txt (51.09 KB)
- bert_lemmatized.txt (45.89 KB)
- attention_is_all_you_need_cleaned.txt (28.79 KB)
- attention_is_all_you_need_normalized.txt (27.76 KB)
- bert_stopword_removed.txt (47.64 KB)
- bert_cleaned.txt (36.65 KB)
- bert.txt (63.49 KB)
- attention_is_all_you_need_lemmatized.txt (20.82 KB)
- attention_is_all_you_need_stopword_removed.txt (21.78 KB)
- rag_normalized.txt (61.87 KB)
- rag_lemmatized.txt (49.37 KB)


The cleaning process applies general rules across all selected academic documents. These include removing page markers, email addresses, URLs, excessive whitespace, front matter before the Abstract section, and the References section when it appears near the end of the document.

The cleaning function is intentionally not over-specialised for one paper. Some document-specific artefacts, such as author contribution notes or publisher-specific formatting, may remain if they appear inside the main extracted text. This is acceptable because overly aggressive cleaning may accidentally remove useful academic or technical content from other documents.

The goal of this step is to reduce major extraction noise while preserving the meaning and structure of the academic documents for later NLP processing, retrieval, and question answering.

## 6.1 Text Cleaning Discussion

After cleaning, the document text becomes more suitable for NLP processing. The cleaning process removes unnecessary page markers, repeated whitespace, extra line breaks, and some formatting noise introduced during PDF extraction.

However, the cleaning process is intentionally not too aggressive. Academic and technical documents often contain important punctuation, mathematical symbols, abbreviations, citations, and section numbering. Removing too much information may damage the meaning of the text.

Therefore, this project applies moderate cleaning at this stage. More task-specific normalization will be performed in the next phase.

## 7. Tokenization

Tokenization is the process of splitting text into smaller meaningful units called tokens.

There are two main types of tokenization used in this project:

1. **Sentence Tokenization**  
   This splits a document into individual sentences.

2. **Word Tokenization**  
   This splits sentences or documents into individual words and symbols.

Tokenization is important because most NLP techniques do not work directly with raw paragraphs. Instead, text must first be broken into smaller units so that it can be analysed, cleaned, encoded, and used for retrieval or question answering.

For academic and technical documents, tokenization helps divide long research papers into manageable units for further processing.

In [121]:
# Import NLTK tokenizers

from nltk.tokenize import sent_tokenize, word_tokenize

# Extra download for compatibility with newer NLTK versions
nltk.download("punkt_tab")

print("Tokenizers are ready.")

Tokenizers are ready.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [122]:
# Select a sample document for tokenization demonstration

sample_document = "attention_is_all_you_need.pdf"

# Use only a portion of the cleaned text for clear demonstration
sample_text = cleaned_texts[sample_document][:3000]

print("Sample text selected from:", sample_document)
print("=" * 80)
print(sample_text[:1000])

Sample text selected from: attention_is_all_you_need.pdf
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training cos

In [123]:
# Perform sentence tokenization

sentence_tokens = sent_tokenize(sample_text)

print("Number of sentence tokens:", len(sentence_tokens))
print("\nSample sentence tokens:")
print("=" * 80)

for i, sentence in enumerate(sentence_tokens[:10], start=1):
    print(f"{i}. {sentence}")

Number of sentence tokens: 22

Sample sentence tokens:
1. Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.
2. The best performing models also connect the encoder and decoder through an attention mechanism.
3. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
4. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train.
5. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU.
6. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of 

In [124]:
# Perform word tokenization

word_tokens = word_tokenize(sample_text)

print("Number of word tokens:", len(word_tokens))
print("\nSample word tokens:")
print("=" * 80)
print(word_tokens[:80])

Number of word tokens: 494

Sample word tokens:
['Abstract', 'The', 'dominant', 'sequence', 'transduction', 'models', 'are', 'based', 'on', 'complex', 'recurrent', 'or', 'convolutional', 'neural', 'networks', 'that', 'include', 'an', 'encoder', 'and', 'a', 'decoder', '.', 'The', 'best', 'performing', 'models', 'also', 'connect', 'the', 'encoder', 'and', 'decoder', 'through', 'an', 'attention', 'mechanism', '.', 'We', 'propose', 'a', 'new', 'simple', 'network', 'architecture', ',', 'the', 'Transformer', ',', 'based', 'solely', 'on', 'attention', 'mechanisms', ',', 'dispensing', 'with', 'recurrence', 'and', 'convolutions', 'entirely', '.', 'Experiments', 'on', 'two', 'machine', 'translation', 'tasks', 'show', 'these', 'models', 'to', 'be', 'superior', 'in', 'quality', 'while', 'being', 'more', 'parallelizable']


In [125]:
# Create tokenization summary for all documents

tokenization_summary = []

for filename, text in cleaned_texts.items():
    sentences = sent_tokenize(text)
    words = word_tokenize(text)

    tokenization_summary.append({
        "File Name": filename,
        "Number of Sentences": len(sentences),
        "Number of Word Tokens": len(words),
        "Average Words per Sentence": round(len(words) / len(sentences), 2) if len(sentences) > 0 else 0
    })

tokenization_summary_df = pd.DataFrame(tokenization_summary)

display(tokenization_summary_df)

,File Name,Number of Sentences,Number of Word Tokens,Average Words per Sentence
0,attention_is_all_you_need.pdf,209,5404,25.86
1,bert.pdf,249,7076,28.42
2,rag.pdf,253,6905,27.29


## 7.1 Sentence Tokenization vs Word Tokenization

Sentence tokenization and word tokenization serve different purposes in NLP.

**Sentence tokenization** is useful when we want to analyse text at the sentence level. For example, it can help in document chunking, summarisation, and identifying meaningful sections of text.

**Word tokenization** is useful when we want to analyse individual words. It is commonly used for stopword removal, stemming, lemmatization, Bag of Words, TF-IDF, and other feature extraction techniques.

In this project, both forms of tokenization are important. Sentence tokenization helps break academic documents into readable units, while word tokenization supports the classical NLP preprocessing pipeline.

In [126]:
# Save sample tokenized output for reference

tokenized_sample = {
    "Sentence Tokens": sentence_tokens[:10],
    "Word Tokens": word_tokens[:100]
}

tokenized_sample_df = pd.DataFrame({
    "Sentence Tokens Sample": pd.Series(tokenized_sample["Sentence Tokens"]),
    "Word Tokens Sample": pd.Series(tokenized_sample["Word Tokens"])
})

display(tokenized_sample_df)

,Sentence Tokens Sample,Word Tokens Sample
0,Abstract The dominant sequence transduction mo...,Abstract
1,The best performing models also connect the en...,The
2,"We propose a new simple network architecture, ...",dominant
3,Experiments on two machine translation tasks s...,sequence
4,Our model achieves 28.4 BLEU on the WMT 2014 E...,transduction
...,...,...
95,NaN,WMT
96,NaN,2014
97,NaN,English-
98,NaN,to-German


## 8. Normalization

Normalization is the process of converting text into a consistent and standard format.

In academic and technical documents, the same word may appear in different forms because of capitalization, punctuation, symbols, or formatting. For example, "Transformer", "transformer", and "TRANSFORMER" may be treated as different tokens by some algorithms if normalization is not applied.

In this project, normalization includes:

1. Converting text to lowercase
2. Removing URLs if present
3. Removing punctuation attached to words
4. Preserving decimal numbers such as 41.8, 92.5, and 0.001
5. Keeping letters, numbers, decimal values, and spaces
6. Standardising extra spaces

For classical NLP processing, punctuation is removed so that words such as "provided", "provided,", and "provided." are treated as the same token.

However, decimal numbers are preserved because academic and technical documents often contain important numerical values such as accuracy scores, percentages, p-values, experiment results, and model metrics.

The cleaned version of the text is still preserved separately for retrieval display and RAG-based answer generation, where punctuation helps preserve readability and meaning.

In [127]:
#Creating the Normalization Function
def normalize_text(text):
    """
    Normalizes cleaned document text for classical NLP processing.

    Normalization steps:
    1. Convert text to lowercase.
    2. Remove URLs if present.
    3. Preserve decimal numbers such as 41.8 and 0.001.
    4. Remove punctuation and special characters attached to words.
    5. Keep alphabets, numbers, decimal values, and spaces.
    6. Standardise spacing.

    Parameters:
    text (str): Cleaned text.

    Returns:
    str: Normalized text.
    """

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs if present
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Temporarily protect decimal points inside numbers
    # Example: 41.8 becomes 41decimalpointtoken8
    text = re.sub(r"(?<=\d)\.(?=\d)", "decimalpointtoken", text)

    # Remove punctuation and special characters
    # Keep only lowercase letters, numbers, and spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Restore decimal points
    text = text.replace("decimalpointtoken", ".")

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing spaces
    text = text.strip()

    return text

In [128]:
# Apply normalization to all cleaned documents

normalized_texts = {}

for filename, text in cleaned_texts.items():
    normalized_texts[filename] = normalize_text(text)

print("Text normalization completed for all documents.")

Text normalization completed for all documents.


In [129]:
# Compare cleaned and normalized text from one sample document

sample_document = "attention_is_all_you_need.pdf"

print("CLEANED TEXT SAMPLE")
print("=" * 80)
print(cleaned_texts[sample_document][:1500])

print("\n\nNORMALIZED TEXT SAMPLE")
print("=" * 80)
print(normalized_texts[sample_document][:1500])

CLEANED TEXT SAMPLE
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the litera

In [130]:
# Demonstrating why careful punctuation handling is useful for technical documents

example_text = "The model achieved 41.8 accuracy. The method provided, useful results with p-value 0.001."

normalized_example = normalize_text(example_text)

print("Original text:")
print(example_text)

print("\nNormalized text:")
print(normalized_example)

print("\nWord tokens after normalization:")
print(normalized_example.split())

Original text:
The model achieved 41.8 accuracy. The method provided, useful results with p-value 0.001.

Normalized text:
the model achieved 41.8 accuracy the method provided useful results with p value 0.001

Word tokens after normalization:
['the', 'model', 'achieved', '41.8', 'accuracy', 'the', 'method', 'provided', 'useful', 'results', 'with', 'p', 'value', '0.001']


In [131]:
# Create summary table showing text size before and after normalization

normalization_summary = []

for filename in cleaned_texts.keys():
    cleaned_characters = len(cleaned_texts[filename])
    normalized_characters = len(normalized_texts[filename])

    cleaned_words = len(cleaned_texts[filename].split())
    normalized_words = len(normalized_texts[filename].split())

    normalization_summary.append({
        "File Name": filename,
        "Cleaned Characters": cleaned_characters,
        "Normalized Characters": normalized_characters,
        "Cleaned Words Approx.": cleaned_words,
        "Normalized Words Approx.": normalized_words,
        "Characters Changed or Removed": cleaned_characters - normalized_characters
    })

normalization_summary_df = pd.DataFrame(normalization_summary)

display(normalization_summary_df)

,File Name,Cleaned Characters,Normalized Characters,Cleaned Words Approx.,Normalized Words Approx.,Characters Changed or Removed
0,attention_is_all_you_need.pdf,29395,28424,4589,4740,971
1,bert.pdf,37140,35580,6029,6340,1560
2,rag.pdf,36788,35536,5681,6139,1252


In [132]:
# Save normalized text files

for filename, text in normalized_texts.items():
    normalized_filename = filename.replace(".pdf", "_normalized.txt")
    normalized_path = f"data/processed_text/{normalized_filename}"

    with open(normalized_path, "w", encoding="utf-8") as file:
        file.write(text)

print("Normalized text files saved successfully.")

Normalized text files saved successfully.


In [133]:
# Verify saved normalized text files

print("Files in data/processed_text:")

for file in os.listdir("data/processed_text"):
    file_path = os.path.join("data/processed_text", file)
    file_size_kb = os.path.getsize(file_path) / 1024
    print(f"- {file} ({file_size_kb:.2f} KB)")

Files in data/processed_text:
- attention_is_all_you_need.txt (38.99 KB)
- bert_normalized.txt (34.75 KB)
- rag_cleaned.txt (36.30 KB)
- rag.txt (68.39 KB)
- rag_stopword_removed.txt (51.09 KB)
- bert_lemmatized.txt (45.89 KB)
- attention_is_all_you_need_cleaned.txt (28.79 KB)
- attention_is_all_you_need_normalized.txt (27.76 KB)
- bert_stopword_removed.txt (47.64 KB)
- bert_cleaned.txt (36.65 KB)
- bert.txt (63.49 KB)
- attention_is_all_you_need_lemmatized.txt (20.82 KB)
- attention_is_all_you_need_stopword_removed.txt (21.78 KB)
- rag_normalized.txt (34.70 KB)
- rag_lemmatized.txt (49.37 KB)


## 8.1 Regarding Normalization

Normalization makes the text more consistent for classical NLP processing. Lowercasing ensures that words such as "Transformer", "transformer", and "TRANSFORMER" are treated as the same word.

Punctuation attached to words is removed during normalization because classical NLP models such as Bag of Words and TF-IDF may otherwise treat words like "provided", "provided,", and "provided." as different tokens. Removing punctuation reduces unnecessary token variation.

However, decimal numbers are preserved because academic and technical documents often contain important numerical values such as accuracy scores, percentages, p-values, experiment results, and model metrics. For example, the value "41.8" should not be converted into separate tokens "41" and "8" because this changes the meaning of the technical content.

Therefore, this project keeps two versions of the text:

1. **Cleaned text**: Used later for document chunking, retrieval display, and RAG context because it preserves readability.
2. **Normalized text**: Used for classical NLP demonstrations such as stopword removal, stemming, lemmatization, Bag of Words, and TF-IDF preprocessing.

This approach balances text simplification with meaning preservation.

## 9. Stopword Removal

Stopwords are common words that appear frequently in a language but often carry limited meaning on their own. Examples include words such as "the", "is", "and", "of", "to", and "in".

In classical NLP tasks, stopword removal is useful because it reduces noise and helps models focus on more meaningful words. For example, in Bag of Words and TF-IDF, removing stopwords can make important technical terms more visible.

However, stopword removal must be used carefully in question-answering systems. Some stopwords can affect the meaning of a question. For example, removing words from a question such as "What is the difference between BERT and RAG?" may reduce its grammatical structure.

Therefore, in this project, stopword removal is demonstrated as part of the classical NLP pipeline, but the final retrieval and RAG stages will use cleaned text chunks to preserve meaning and readability.

In [134]:
# Load English stopwords from NLTK

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print("Number of English stopwords:", len(stop_words))
print("\nSample stopwords:")
print(list(stop_words)[:30])

Number of English stopwords: 198

Sample stopwords:
["won't", 'ours', "it'll", 'of', 'had', 'should', 'ourselves', "shouldn't", 'shouldn', 'we', "he'll", 'up', 'couldn', "weren't", 'can', 'won', "you'd", 'before', 'ain', "you've", 'yourselves', 'have', "we're", 'that', 'where', 'my', 'theirs', 'here', "hasn't", 'o']


In [135]:
#Stopword Removal Function
def remove_stopwords(text):
    """
    Removes common English stopwords from normalized text.

    Parameters:
    text (str): Normalized text.

    Returns:
    str: Text after stopword removal.
    """

    # Split normalized text into tokens
    tokens = text.split()

    # Remove stopwords
    filtered_tokens = [word for word in tokens if word not in stop_words]

    # Join tokens back into text
    filtered_text = " ".join(filtered_tokens)

    return filtered_text

In [136]:
# Apply stopword removal to all normalized documents

stopword_removed_texts = {}

for filename, text in normalized_texts.items():
    stopword_removed_texts[filename] = remove_stopwords(text)

print("Stopword removal completed for all documents.")

Stopword removal completed for all documents.


In [137]:
# Compare normalized text and stopword-removed text

sample_document = "attention_is_all_you_need.pdf"

print("NORMALIZED TEXT SAMPLE")
print("=" * 80)
print(normalized_texts[sample_document][:1200])

print("\n\nTEXT AFTER STOPWORD REMOVAL")
print("=" * 80)
print(stopword_removed_texts[sample_document][:1200])

NORMALIZED TEXT SAMPLE
abstract the dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder the best performing models also connect the encoder and decoder through an attention mechanism we propose a new simple network architecture the transformer based solely on attention mechanisms dispensing with recurrence and convolutions entirely experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train our model achieves 28.4 bleu on the wmt 2014 english to german translation task improving over the existing best results including ensembles by over 2 bleu on the wmt 2014 english to french translation task our model establishes a new single model state of the art bleu score of 41.8 after training for 3.5 days on eight gpus a small fraction of the training costs of the best models from the literature we sho

In [138]:
# Demonstrate stopword removal with a simple example

example_sentence = "The transformer model uses attention mechanisms to process input sequences efficiently."

normalized_example = normalize_text(example_sentence)
stopword_removed_example = remove_stopwords(normalized_example)

print("Original sentence:")
print(example_sentence)

print("\nNormalized sentence:")
print(normalized_example)

print("\nAfter stopword removal:")
print(stopword_removed_example)

Original sentence:
The transformer model uses attention mechanisms to process input sequences efficiently.

Normalized sentence:
the transformer model uses attention mechanisms to process input sequences efficiently

After stopword removal:
transformer model uses attention mechanisms process input sequences efficiently


In [139]:
# Creating summary table showing text size before and after stopword removal

stopword_summary = []

for filename in normalized_texts.keys():
    normalized_words = len(normalized_texts[filename].split())
    filtered_words = len(stopword_removed_texts[filename].split())

    stopword_summary.append({
        "File Name": filename,
        "Words Before Stopword Removal": normalized_words,
        "Words After Stopword Removal": filtered_words,
        "Stopwords Removed Approx.": normalized_words - filtered_words,
        "Reduction Percentage": round(((normalized_words - filtered_words) / normalized_words) * 100, 2)
    })

stopword_summary_df = pd.DataFrame(stopword_summary)

display(stopword_summary_df)

,File Name,Words Before Stopword Removal,Words After Stopword Removal,Stopwords Removed Approx.,Reduction Percentage
0,attention_is_all_you_need.pdf,4740,3096,1644,34.68
1,bert.pdf,6340,4315,2025,31.94
2,rag.pdf,6139,4129,2010,32.74


In [140]:
# Show which stopwords were removed from a sample sentence

example_sentence = "The RAG model retrieves relevant document chunks before generating an answer."

normalized_example = normalize_text(example_sentence)
tokens = normalized_example.split()

removed_words = [word for word in tokens if word in stop_words]
remaining_words = [word for word in tokens if word not in stop_words]

print("Original normalized tokens:")
print(tokens)

print("\nRemoved stopwords:")
print(removed_words)

print("\nRemaining words:")
print(remaining_words)

Original normalized tokens:
['the', 'rag', 'model', 'retrieves', 'relevant', 'document', 'chunks', 'before', 'generating', 'an', 'answer']

Removed stopwords:
['the', 'before', 'an']

Remaining words:
['rag', 'model', 'retrieves', 'relevant', 'document', 'chunks', 'generating', 'answer']


In [141]:
# Save stopword-removed text files

for filename, text in stopword_removed_texts.items():
    stopword_filename = filename.replace(".pdf", "_stopword_removed.txt")
    stopword_path = f"data/processed_text/{stopword_filename}"

    with open(stopword_path, "w", encoding="utf-8") as file:
        file.write(text)

print("Stopword-removed text files saved successfully.")

Stopword-removed text files saved successfully.


In [142]:
# Verify saved stopword-removed text files

print("Files in data/processed_text:")

for file in os.listdir("data/processed_text"):
    file_path = os.path.join("data/processed_text", file)
    file_size_kb = os.path.getsize(file_path) / 1024
    print(f"- {file} ({file_size_kb:.2f} KB)")

Files in data/processed_text:
- attention_is_all_you_need.txt (38.99 KB)
- bert_normalized.txt (34.75 KB)
- rag_cleaned.txt (36.30 KB)
- rag.txt (68.39 KB)
- rag_stopword_removed.txt (27.34 KB)
- bert_lemmatized.txt (45.89 KB)
- attention_is_all_you_need_cleaned.txt (28.79 KB)
- attention_is_all_you_need_normalized.txt (27.76 KB)
- bert_stopword_removed.txt (27.42 KB)
- bert_cleaned.txt (36.65 KB)
- bert.txt (63.49 KB)
- attention_is_all_you_need_lemmatized.txt (20.82 KB)
- attention_is_all_you_need_stopword_removed.txt (21.78 KB)
- rag_normalized.txt (34.70 KB)
- rag_lemmatized.txt (49.37 KB)


## 9.1 Regarding Stopword Removal

Stopword removal reduces the number of common words in the document text. This helps classical NLP techniques focus more on meaningful terms such as "transformer", "attention", "retrieval", "generation", "embedding", and "model".

This step is especially useful for keyword-based representations such as Bag of Words and TF-IDF because frequent but less informative words can otherwise dominate the representation.

However, stopword removal may not always be suitable for question-answering. In QA tasks, small words can help preserve the meaning and structure of a question. For example, words such as "not", "between", "before", and "after" can affect the meaning of a sentence.

Therefore, this project demonstrates stopword removal as part of the classical NLP pipeline, but the later retrieval and RAG stages will use readable cleaned text chunks rather than heavily reduced stopword-removed text.

## 10. Stemming and Lemmatization

Stemming and lemmatization are text normalization techniques used to reduce words to their base or root form.

**Stemming** removes word endings using rule-based methods. For example, "studying" may become "studi" and "generating" may become "gener". This is fast, but it can produce incomplete or non-dictionary words.

**Lemmatization** reduces words to their proper dictionary form, called a lemma. For example, "studying" becomes "study" and "generating" becomes "generate".

However, lemmatization works best when part-of-speech information is available. Without POS information, some lemmatizers assume words are nouns by default, which may lead to incomplete lemmatization of verbs.

In academic and technical documents, preserving meaningful and readable terms is important. Therefore, lemmatization is more suitable for this project than stemming.

In [143]:
# Import stemming and lemmatization tools

from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

print("Stemmer and lemmatizer loaded successfully.")

Stemmer and lemmatizer loaded successfully.


In [144]:
# Compare stemming and lemmatization using sample words

sample_words = [
    "running",
    "runs",
    "studies",
    "studying",
    "retrieved",
    "retrieval",
    "generating",
    "generated",
    "models",
    "transformers",
    "better"
]

comparison_data = []

for word in sample_words:
    comparison_data.append({
        "Original Word": word,
        "Stemmed Form": stemmer.stem(word),
        "NLTK Lemmatized Without POS": lemmatizer.lemmatize(word),
        "NLTK Lemmatized As Verb": lemmatizer.lemmatize(word, pos="v")
    })

comparison_df = pd.DataFrame(comparison_data)

display(comparison_df)

,Original Word,Stemmed Form,NLTK Lemmatized Without POS,NLTK Lemmatized As Verb
0,running,run,running,run
1,runs,run,run,run
2,studies,studi,study,study
3,studying,studi,studying,study
4,retrieved,retriev,retrieved,retrieve
5,retrieval,retriev,retrieval,retrieval
6,generating,gener,generating,generate
7,generated,gener,generated,generate
8,models,model,model,model
9,transformers,transform,transformer,transformers


## 10.1 Observation from Stemming and Lemmatization Comparison

The comparison table shows that stemming and lemmatization behave differently.

Stemming reduces words by cutting suffixes. This can produce shortened forms such as "studi", "retriev", and "gener", which are not proper dictionary words.

Basic NLTK lemmatization without POS information is less aggressive because it assumes words are nouns by default. Therefore, some verb forms such as "running", "studying", and "generating" may remain unchanged.

When POS information is provided, lemmatization becomes more accurate. For example, when words are treated as verbs, "running" becomes "run", "studying" becomes "study", and "generating" becomes "generate".

This demonstrates why POS-aware lemmatization is more suitable for academic and technical document processing.

In [145]:
# Demonstrate lemmatization with spaCy POS information

example_sentence = "The models were generating useful answers and retrieving relevant document chunks."

doc = nlp(example_sentence)

lemma_data = []

for token in doc:
    lemma_data.append({
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS Tag": token.pos_
    })

lemma_df = pd.DataFrame(lemma_data)

display(lemma_df)

,Token,Lemma,POS Tag
0,The,the,DET
1,models,model,NOUN
2,were,be,AUX
3,generating,generate,VERB
4,useful,useful,ADJ
5,answers,answer,NOUN
6,and,and,CCONJ
7,retrieving,retrieve,VERB
8,relevant,relevant,ADJ
9,document,document,NOUN


In [146]:
#spaCy Lemmatization Function
def lemmatize_text(text):
    """
    Lemmatizes normalized and stopword-removed text using spaCy.

    Parameters:
    text (str): Text after normalization and stopword removal.

    Returns:
    str: Lemmatized text.
    """

    doc = nlp(text)

    lemmas = []

    for token in doc:
        if not token.is_space:
            lemmas.append(token.lemma_)

    return " ".join(lemmas)

In [147]:
# Apply lemmatization to all stopword-removed documents

lemmatized_texts = {}

for filename, text in stopword_removed_texts.items():
    print(f"Lemmatizing: {filename}")
    lemmatized_texts[filename] = lemmatize_text(text)

print("Lemmatization completed for all documents.")

Lemmatizing: attention_is_all_you_need.pdf
Lemmatizing: bert.pdf
Lemmatizing: rag.pdf
Lemmatization completed for all documents.


In [148]:
# Compare stopword-removed text and lemmatized text

sample_document = "attention_is_all_you_need.pdf"

print("TEXT AFTER STOPWORD REMOVAL")
print("=" * 80)
print(stopword_removed_texts[sample_document][:1200])

print("\n\nLEMMATIZED TEXT")
print("=" * 80)
print(lemmatized_texts[sample_document][:1200])

TEXT AFTER STOPWORD REMOVAL
abstract dominant sequence transduction models based complex recurrent convolutional neural networks include encoder decoder best performing models also connect encoder decoder attention mechanism propose new simple network architecture transformer based solely attention mechanisms dispensing recurrence convolutions entirely experiments two machine translation tasks show models superior quality parallelizable requiring significantly less time train model achieves 28.4 bleu wmt 2014 english german translation task improving existing best results including ensembles 2 bleu wmt 2014 english french translation task model establishes new single model state art bleu score 41.8 training 3.5 days eight gpus small fraction training costs best models literature show transformer generalizes well tasks applying successfully english constituency parsing large limited training data equal contribution listing order random jakob proposed replacing rnns self attention starte

In [149]:
# Demonstrate stemming and lemmatization on a short sample text

sample_sentence = "The models were generating useful answers and retrieving relevant document chunks."

normalized_sample = normalize_text(sample_sentence)
tokens = normalized_sample.split()

stemmed_tokens = [stemmer.stem(word) for word in tokens]

spacy_doc = nlp(normalized_sample)
lemmatized_tokens = [token.lemma_ for token in spacy_doc if not token.is_space]

comparison_sample_df = pd.DataFrame({
    "Original Token": tokens,
    "Stemmed Token": stemmed_tokens,
    "spaCy Lemmatized Token": lemmatized_tokens
})

display(comparison_sample_df)

,Original Token,Stemmed Token,spaCy Lemmatized Token
0,the,the,the
1,models,model,model
2,were,were,be
3,generating,gener,generate
4,useful,use,useful
5,answers,answer,answer
6,and,and,and
7,retrieving,retriev,retrieve
8,relevant,relev,relevant
9,document,document,document


In [150]:
# Create summary table showing text size before and after lemmatization

lemmatization_summary = []

for filename in stopword_removed_texts.keys():
    words_before = len(stopword_removed_texts[filename].split())
    words_after = len(lemmatized_texts[filename].split())

    lemmatization_summary.append({
        "File Name": filename,
        "Words Before Lemmatization": words_before,
        "Words After Lemmatization": words_after,
        "Word Count Difference": words_before - words_after
    })

lemmatization_summary_df = pd.DataFrame(lemmatization_summary)

display(lemmatization_summary_df)

,File Name,Words Before Lemmatization,Words After Lemmatization,Word Count Difference
0,attention_is_all_you_need.pdf,3096,3098,-2
1,bert.pdf,4315,4332,-17
2,rag.pdf,4129,4137,-8


The word count after lemmatization may slightly differ from the word count before lemmatization because spaCy performs token-based processing. Some technical terms, hyphenated words, abbreviations, or special tokens may be split or represented differently during lemmatization. Therefore, lemmatization mainly changes word forms, but small changes in token count are expected.

In [151]:
# Save lemmatized text files

for filename, text in lemmatized_texts.items():
    lemmatized_filename = filename.replace(".pdf", "_lemmatized.txt")
    lemmatized_path = f"data/processed_text/{lemmatized_filename}"

    with open(lemmatized_path, "w", encoding="utf-8") as file:
        file.write(text)

print("Lemmatized text files saved successfully.")

Lemmatized text files saved successfully.


In [152]:
# Verify saved lemmatized text files

print("Files in data/processed_text:")

for file in os.listdir("data/processed_text"):
    file_path = os.path.join("data/processed_text", file)
    file_size_kb = os.path.getsize(file_path) / 1024
    print(f"- {file} ({file_size_kb:.2f} KB)")

Files in data/processed_text:
- attention_is_all_you_need.txt (38.99 KB)
- bert_normalized.txt (34.75 KB)
- rag_cleaned.txt (36.30 KB)
- rag.txt (68.39 KB)
- rag_stopword_removed.txt (27.34 KB)
- bert_lemmatized.txt (26.36 KB)
- attention_is_all_you_need_cleaned.txt (28.79 KB)
- attention_is_all_you_need_normalized.txt (27.76 KB)
- bert_stopword_removed.txt (27.42 KB)
- bert_cleaned.txt (36.65 KB)
- bert.txt (63.49 KB)
- attention_is_all_you_need_lemmatized.txt (20.82 KB)
- attention_is_all_you_need_stopword_removed.txt (21.78 KB)
- rag_normalized.txt (34.70 KB)
- rag_lemmatized.txt (26.24 KB)


## 10.2 Stemming vs Lemmatization Discussion

Stemming and lemmatization both reduce words to simpler forms, but they differ in accuracy and readability.

Stemming is fast and simple, but it often produces incomplete word forms. For example, "studying" may become "studi" and "generating" may become "gener". These forms may still be useful for keyword matching, but they are less readable and less suitable for explainable NLP systems.

Lemmatization is more linguistically accurate because it reduces words to proper dictionary forms. For example, "generating" becomes "generate", "retrieving" becomes "retrieve", and "models" becomes "model".

The comparison also shows that lemmatization depends on part-of-speech information. Without POS information, some lemmatizers assume words are nouns by default, which may leave verb forms unchanged. POS-aware lemmatization, such as spaCy lemmatization, uses grammatical context to produce better base forms.

For this project, lemmatization is preferred because the system is designed for academic and technical document question answering, where readability and meaning preservation are important.

Stemming was demonstrated for comparison, but it was not applied to the full document collection because it can produce incomplete and less interpretable word forms. Since this project focuses on explainable question answering over academic and technical documents, spaCy-based lemmatization was applied to the documents instead.

Therefore, stemming is demonstrated for learning purposes, while lemmatization is used as the preferred method in the classical NLP preprocessing pipeline.

## 11. Part-of-Speech Tagging

Part-of-Speech tagging, also called POS tagging, is the process of identifying the grammatical role of each word in a sentence.

For example, words can be classified as nouns, verbs, adjectives, adverbs, pronouns, prepositions, or other grammatical categories.

Although spaCy already uses part-of-speech information internally during lemmatization, POS tagging is shown separately in this project because it is an important NLP concept on its own.

In the lemmatization phase, POS information helped produce better base word forms. In this phase, POS tagging is demonstrated explicitly to show how words are classified into grammatical categories.

This makes the linguistic analysis process more transparent and satisfies the classical NLP pipeline requirement of the project.

In academic and technical documents:

- Nouns often represent key concepts, models, methods, datasets, or technical terms.
- Verbs often represent actions or processes.
- Adjectives often describe properties or qualities.
- Adverbs often describe how an action is performed.

POS tagging helps us understand the linguistic structure of documents before moving into encoding, retrieval, and question answering.

In [153]:
# POS tagging on a clean technical sentence

sample_sentence = "The Transformer model uses self-attention mechanisms to process sequential data efficiently."

doc = nlp(sample_sentence)

pos_data = []

for token in doc:
    pos_data.append({
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS Tag": token.pos_,
        "Detailed Tag": token.tag_,
        "POS Explanation": spacy.explain(token.pos_),
        "Detailed Tag Explanation": spacy.explain(token.tag_)
    })

pos_df = pd.DataFrame(pos_data)

display(pos_df)

,Token,Lemma,POS Tag,Detailed Tag,POS Explanation,Detailed Tag Explanation
0,The,the,DET,DT,determiner,determiner
1,Transformer,transformer,ADJ,JJ,adjective,"adjective (English), other noun-modifier (Chin..."
2,model,model,NOUN,NN,noun,"noun, singular or mass"
3,uses,use,VERB,VBZ,verb,"verb, 3rd person singular present"
4,self,self,NOUN,NN,noun,"noun, singular or mass"
5,-,-,PUNCT,HYPH,punctuation,"punctuation mark, hyphen"
6,attention,attention,NOUN,NN,noun,"noun, singular or mass"
7,mechanisms,mechanism,NOUN,NNS,noun,"noun, plural"
8,to,to,PART,TO,particle,"infinitival ""to"""
9,process,process,VERB,VB,verb,"verb, base form"


In [154]:
# Extract important POS categories from the sample sentence

important_pos = {
    "Nouns / Proper Nouns": [],
    "Verbs": [],
    "Adjectives": [],
    "Adverbs": []
}

for token in doc:
    if token.pos_ in ["NOUN", "PROPN"]:
        important_pos["Nouns / Proper Nouns"].append(token.text)
    elif token.pos_ == "VERB":
        important_pos["Verbs"].append(token.text)
    elif token.pos_ == "ADJ":
        important_pos["Adjectives"].append(token.text)
    elif token.pos_ == "ADV":
        important_pos["Adverbs"].append(token.text)

important_pos_df = pd.DataFrame(dict([
    (key, pd.Series(value)) for key, value in important_pos.items()
]))

display(important_pos_df)

,Nouns / Proper Nouns,Verbs,Adjectives,Adverbs
0,model,uses,Transformer,efficiently
1,self,process,sequential,NaN
2,attention,NaN,NaN,NaN
3,mechanisms,NaN,NaN,NaN
4,data,NaN,NaN,NaN


In [155]:
# Select and clean a document sample before POS tagging

sample_document = "attention_is_all_you_need.pdf"

# Use a limited sample for clear display
document_sample = cleaned_texts[sample_document][:3000]

# Remove email addresses because they are not normal sentence tokens
document_sample = re.sub(r"\S+@\S+", " ", document_sample)

# Remove URLs if present
document_sample = re.sub(r"http\S+|www\S+", " ", document_sample)

# Remove extra spaces after filtering
document_sample = re.sub(r"\s+", " ", document_sample).strip()

print("Cleaned document sample for POS tagging:")
print("=" * 80)
print(document_sample[:1000])

Cleaned document sample for POS tagging:
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best m

In [156]:
# Apply POS tagging to the selected document sample

document_doc = nlp(document_sample)

document_pos_data = []

for token in document_doc:
    if not token.is_space and not token.is_punct:
        document_pos_data.append({
            "Token": token.text,
            "Lemma": token.lemma_,
            "POS Tag": token.pos_,
            "Detailed Tag": token.tag_,
            "POS Explanation": spacy.explain(token.pos_)
        })

document_pos_df = pd.DataFrame(document_pos_data)

display(document_pos_df.head(40))

,Token,Lemma,POS Tag,Detailed Tag,POS Explanation
0,Abstract,abstract,ADV,RB,adverb
1,The,the,DET,DT,determiner
2,dominant,dominant,ADJ,JJ,adjective
3,sequence,sequence,NOUN,NN,noun
4,transduction,transduction,NOUN,NN,noun
5,models,model,NOUN,NNS,noun
6,are,be,AUX,VBP,auxiliary
7,based,base,VERB,VBN,verb
8,on,on,ADP,IN,adposition
9,complex,complex,ADJ,JJ,adjective


In [157]:
# Count POS tag frequencies in the document sample

pos_frequency = document_pos_df["POS Tag"].value_counts().reset_index()
pos_frequency.columns = ["POS Tag", "Frequency"]

display(pos_frequency)

,POS Tag,Frequency
0,NOUN,130
1,VERB,53
2,ADJ,50
3,ADP,50
4,DET,35
5,PROPN,35
6,CCONJ,24
7,NUM,19
8,ADV,17
9,PRON,14


In [158]:
# Extract nouns, verbs, adjectives, and adverbs from the document sample

nouns = []
verbs = []
adjectives = []
adverbs = []

for token in document_doc:
    if not token.is_space and not token.is_punct:
        if token.pos_ in ["NOUN", "PROPN"]:
            nouns.append(token.text)
        elif token.pos_ == "VERB":
            verbs.append(token.text)
        elif token.pos_ == "ADJ":
            adjectives.append(token.text)
        elif token.pos_ == "ADV":
            adverbs.append(token.text)

pos_category_df = pd.DataFrame({
    "Nouns / Proper Nouns": pd.Series(nouns[:25]),
    "Verbs": pd.Series(verbs[:25]),
    "Adjectives": pd.Series(adjectives[:25]),
    "Adverbs": pd.Series(adverbs[:25])
})

display(pos_category_df)

,Nouns / Proper Nouns,Verbs,Adjectives,Adverbs
0,sequence,based,dominant,Abstract
1,transduction,include,complex,best
2,models,performing,convolutional,also
3,recurrent,connect,neural,solely
4,networks,propose,new,entirely
5,encoder,based,simple,more
6,decoder,dispensing,superior,significantly
7,models,show,parallelizable,well
8,encoder,requiring,less,successfully
9,decoder,train,German,crucially


In [159]:
# POS tagging summary for all documents using limited samples

pos_summary = []

for filename, text in cleaned_texts.items():
    print(f"Processing POS tags for: {filename}")

    # Use a limited sample to keep the notebook readable and efficient
    sample_text = text[:5000]

    # Remove emails and URLs if any remain
    sample_text = re.sub(r"\S+@\S+", " ", sample_text)
    sample_text = re.sub(r"http\S+|www\S+", " ", sample_text)
    sample_text = re.sub(r"\s+", " ", sample_text).strip()

    sample_doc = nlp(sample_text)

    noun_count = 0
    verb_count = 0
    adjective_count = 0
    adverb_count = 0

    for token in sample_doc:
        if not token.is_space and not token.is_punct:
            if token.pos_ in ["NOUN", "PROPN"]:
                noun_count += 1
            elif token.pos_ == "VERB":
                verb_count += 1
            elif token.pos_ == "ADJ":
                adjective_count += 1
            elif token.pos_ == "ADV":
                adverb_count += 1

    pos_summary.append({
        "File Name": filename,
        "Sample Used": "First 5000 characters",
        "Nouns / Proper Nouns": noun_count,
        "Verbs": verb_count,
        "Adjectives": adjective_count,
        "Adverbs": adverb_count
    })

pos_summary_df = pd.DataFrame(pos_summary)

display(pos_summary_df)

Processing POS tags for: attention_is_all_you_need.pdf
Processing POS tags for: bert.pdf
Processing POS tags for: rag.pdf


,File Name,Sample Used,Nouns / Proper Nouns,Verbs,Adjectives,Adverbs
0,attention_is_all_you_need.pdf,First 5000 characters,266,83,81,28
1,bert.pdf,First 5000 characters,313,88,89,25
2,rag.pdf,First 5000 characters,300,86,99,21


## 11.1 Why POS Tagging Uses a Limited Sample

POS tagging was applied to selected document samples rather than the entire document collection because this step is used to demonstrate linguistic analysis, not as the final retrieval method.

Full academic papers contain thousands of tokens. Displaying POS tags for every token would make the notebook difficult to read and would not add much value to the final question-answering system.

In addition, PDF-extracted research papers may contain noisy elements such as author contribution notes, affiliations, references, equations, tables, and formatting artefacts. These elements can reduce POS tagging accuracy because POS taggers are mainly designed for natural language sentences.

Therefore, this project uses clean and limited samples to demonstrate POS tagging clearly, while summary counts are used to show POS patterns across the selected documents.

## 11.2 POS Tagging Limitations

POS tagging is useful for understanding grammatical structure, but it is not always perfect.

Academic PDFs often contain non-standard text such as citations, equations, references, section numbers, author affiliations, and occasional front-matter notes. These are not always normal sentence tokens, so POS taggers may classify them incorrectly.

For example, an email address, citation marker, or extracted formatting artefact may receive an incorrect POS tag because the tagger tries to interpret it based on surrounding text. This does not mean the code is wrong. It shows a limitation of applying POS tagging directly to PDF-extracted academic text.

To reduce this issue, this project removes emails and URLs before POS tagging document samples and uses limited samples for clearer demonstration.

## 11.3 POS Tagging Discussion

POS tagging helps identify the grammatical role of words in document text. This is useful because academic and technical documents often contain dense information, specialised terminology, and complex sentence structures.

In this project, POS tagging shows how the documents are made up of nouns, verbs, adjectives, and adverbs. Nouns and proper nouns are especially important because they often represent technical concepts such as "Transformer", "attention", "model", "retrieval", and "generation". Verbs help identify actions or processes, while adjectives describe the characteristics of methods, models, or results.

Although POS tagging is not directly used as the final retrieval method in this project, it demonstrates an important classical NLP technique. It also helps explain how text can be analysed linguistically before applying machine learning, embeddings, or Retrieval-Augmented Generation.

## 12. Dependency Parsing

Dependency parsing is an NLP technique used to identify grammatical relationships between words in a sentence.

While POS tagging tells us the grammatical category of each word, dependency parsing shows how words are connected to each other.

For example, in the sentence:

"The Transformer model uses attention mechanisms."

Dependency parsing can show that:

- "uses" is the main verb
- "model" is the subject of "uses"
- "mechanisms" is the object of "uses"
- "Transformer" describes the model
- "attention" describes the mechanisms

This is useful because academic and technical documents often contain complex sentences. Dependency parsing helps us understand how concepts, actions, and descriptions are connected.

In this project, dependency parsing is demonstrated as part of the classical NLP pipeline before moving into encoding, retrieval, and RAG.

In [160]:
# Dependency parsing on a clean technical sentence

sample_sentence = "The Transformer model uses self-attention mechanisms to process sequential data efficiently."

parsed_doc = nlp(sample_sentence)

dependency_data = []

for token in parsed_doc:
    dependency_data.append({
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS Tag": token.pos_,
        "Dependency": token.dep_,
        "Dependency Explanation": spacy.explain(token.dep_),
        "Head Word": token.head.text,
        "Head POS": token.head.pos_
    })

dependency_df = pd.DataFrame(dependency_data)

display(dependency_df)

,Token,Lemma,POS Tag,Dependency,Dependency Explanation,Head Word,Head POS
0,The,the,DET,det,determiner,model,NOUN
1,Transformer,transformer,ADJ,amod,adjectival modifier,model,NOUN
2,model,model,NOUN,nsubj,nominal subject,uses,VERB
3,uses,use,VERB,ROOT,root,uses,VERB
4,self,self,NOUN,compound,compound,attention,NOUN
5,-,-,PUNCT,punct,punctuation,attention,NOUN
6,attention,attention,NOUN,compound,compound,mechanisms,NOUN
7,mechanisms,mechanism,NOUN,dobj,direct object,uses,VERB
8,to,to,PART,aux,auxiliary,process,VERB
9,process,process,VERB,xcomp,open clausal complement,uses,VERB


In [161]:
# Display important dependency relationships in a readable format

print("Dependency relationships:")
print("=" * 80)

for token in parsed_doc:
    print(f"{token.text:<15} --> {token.dep_:<12} --> Head: {token.head.text}")

Dependency relationships:
The             --> det          --> Head: model
Transformer     --> amod         --> Head: model
model           --> nsubj        --> Head: uses
uses            --> ROOT         --> Head: uses
self            --> compound     --> Head: attention
-               --> punct        --> Head: attention
attention       --> compound     --> Head: mechanisms
mechanisms      --> dobj         --> Head: uses
to              --> aux          --> Head: process
process         --> xcomp        --> Head: uses
sequential      --> amod         --> Head: data
data            --> dobj         --> Head: process
efficiently     --> advmod       --> Head: process
.               --> punct        --> Head: uses


In [162]:
# Extract simple subject, verb, and object-style relationships

svo_data = []

for token in parsed_doc:
    if token.dep_ in ["nsubj", "nsubjpass", "dobj", "pobj", "attr"]:
        svo_data.append({
            "Token": token.text,
            "Dependency Role": token.dep_,
            "Role Explanation": spacy.explain(token.dep_),
            "Connected To / Head": token.head.text
        })

svo_df = pd.DataFrame(svo_data)

display(svo_df)

,Token,Dependency Role,Role Explanation,Connected To / Head
0,model,nsubj,nominal subject,uses
1,mechanisms,dobj,direct object,uses
2,data,dobj,direct object,process


In [163]:
# Select a short document sample for dependency parsing

sample_document = "attention_is_all_you_need.pdf"

# Use a smaller sample because dependency parsing output can become large
document_sample_for_parsing = cleaned_texts[sample_document][:1200]

# Remove emails and URLs if any remain
document_sample_for_parsing = re.sub(r"\S+@\S+", " ", document_sample_for_parsing)
document_sample_for_parsing = re.sub(r"http\S+|www\S+", " ", document_sample_for_parsing)
document_sample_for_parsing = re.sub(r"\s+", " ", document_sample_for_parsing).strip()

print("Document sample selected for dependency parsing:")
print("=" * 80)
print(document_sample_for_parsing)

Document sample selected for dependency parsing:
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of th

In [164]:
# Apply dependency parsing to the document sample

document_parsed_doc = nlp(document_sample_for_parsing)

document_dependency_data = []

for token in document_parsed_doc:
    if not token.is_space and not token.is_punct:
        document_dependency_data.append({
            "Token": token.text,
            "Lemma": token.lemma_,
            "POS Tag": token.pos_,
            "Dependency": token.dep_,
            "Dependency Explanation": spacy.explain(token.dep_),
            "Head Word": token.head.text
        })

document_dependency_df = pd.DataFrame(document_dependency_data)

display(document_dependency_df.head(40))

,Token,Lemma,POS Tag,Dependency,Dependency Explanation,Head Word
0,Abstract,abstract,ADV,advmod,adverbial modifier,based
1,The,the,DET,det,determiner,models
2,dominant,dominant,ADJ,amod,adjectival modifier,models
3,sequence,sequence,NOUN,compound,compound,transduction
4,transduction,transduction,NOUN,compound,compound,models
5,models,model,NOUN,nsubjpass,nominal subject (passive),based
6,are,be,AUX,auxpass,auxiliary (passive),based
7,based,base,VERB,ROOT,root,based
8,on,on,ADP,prep,prepositional modifier,based
9,complex,complex,ADJ,amod,adjectival modifier,recurrent


In [165]:
# Count dependency label frequencies in the document sample

dependency_frequency = document_dependency_df["Dependency"].value_counts().reset_index()
dependency_frequency.columns = ["Dependency Label", "Frequency"]

display(dependency_frequency)

,Dependency Label,Frequency
0,prep,24
1,pobj,23
2,amod,19
3,det,19
4,compound,13
5,ROOT,10
6,nsubj,10
7,advmod,9
8,dobj,8
9,conj,7


In [166]:
# Show a few parsed sentences from the document sample

sentences_from_sample = list(document_parsed_doc.sents)

print("Sample sentences identified by spaCy:")
print("=" * 80)

for i, sentence in enumerate(sentences_from_sample[:5], start=1):
    print(f"{i}. {sentence.text}")

Sample sentences identified by spaCy:
1. Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.
2. The best performing models also connect the encoder and decoder through an attention mechanism.
3. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.
4. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train.
5. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU.


## 12.1 Dependency Parsing Discussion

Dependency parsing provides more structural information than POS tagging.

POS tagging identifies the grammatical category of each word, such as noun, verb, adjective, or adverb. Dependency parsing goes further by identifying how words are related to each other in a sentence.

For academic and technical documents, this is useful because important information is often expressed through relationships between concepts. For example, parsing can help identify what a model does, what method is being used, and what object or concept is affected by an action.

In this project, dependency parsing is demonstrated to show how NLP can analyse sentence structure before moving into numerical text representation methods such as Bag of Words, TF-IDF, and embeddings.

However, dependency parsing is not used as the main retrieval method in this project. It is shown as part of the classical NLP pipeline to demonstrate linguistic understanding.

## 12.2 Dependency Parsing Limitations

Dependency parsing can be affected by the quality of extracted text.

Academic PDFs may contain equations, citations, section numbers, author notes, tables, and formatting artefacts. These elements may not follow normal sentence structure, so dependency parsers may sometimes produce incorrect relationships.

For this reason, this project demonstrates dependency parsing on selected readable samples rather than parsing and displaying the full document collection.

In this project, different versions of the document text are maintained for different NLP purposes. The `cleaned_texts` version is used for POS tagging, dependency parsing, document chunking, sentence embeddings, and RAG because it preserves readable sentence structure, punctuation, and overall context. This makes it more suitable for grammar-based and meaning-based tasks. The `normalized_texts` version is used for stopword removal and classical preprocessing demonstrations because it is standardised and simplified. The `lemmatized_texts` version is used for Bag of Words and TF-IDF demonstrations because it reduces words to meaningful base forms and improves keyword consistency. Maintaining multiple versions helps balance readability, linguistic analysis, and computational consistency.

## 13. Encoding Techniques

Encoding is the process of converting text into numerical representations that can be processed by machine learning models and retrieval systems.

In this project, three encoding techniques are demonstrated:

1. **Bag of Words**
   - Represents text using word frequency.
   - It counts how often each word appears in a sentence or document.
   - It is simple and interpretable, but it ignores word order and meaning.

2. **TF-IDF**
   - Stands for Term Frequency-Inverse Document Frequency.
   - It gives higher importance to words that are frequent in one text but not common across all texts.
   - It is useful for keyword-based document retrieval.

3. **Sentence Embeddings**
   - Represents sentences as dense numerical vectors.
   - These vectors capture semantic meaning.
   - Embeddings are useful for semantic search and RAG-based question answering.

Different text versions are used for different encoding techniques. Bag of Words and TF-IDF use preprocessed text because they benefit from normalization, stopword removal, and lemmatization. Sentence embeddings use cleaned readable text because embeddings rely on natural sentence context and meaning.

## 13.1 Text Versions Used for Encoding

In this phase, two types of sample text are prepared.

For **Bag of Words** and **TF-IDF**, the sample sentences are first normalized, stopwords are removed, and lemmatization is applied. This reduces unnecessary variation in word forms and makes keyword-based representations more consistent.

For **sentence embeddings**, cleaned text is used directly because embeddings work better when the sentence remains readable and natural. Punctuation, grammar, and sentence structure help preserve meaning.

Therefore:

- `processed_classical_corpus` is used for Bag of Words and TF-IDF.
- `embedding_corpus` is used for sentence embeddings.

In [167]:
# Prepare sample sentences for encoding demonstration

encoding_samples = []

for filename, text in cleaned_texts.items():
    sentences = sent_tokenize(text)

    # Select readable sentences for demonstration
    selected_sentences = []

    for sent in sentences:
        word_count = len(sent.split())

        # Keep medium-length readable sentences
        if 10 <= word_count <= 60:
            selected_sentences.append(sent)

        if len(selected_sentences) == 3:
            break

    for sentence in selected_sentences:
        encoding_samples.append({
            "Document": filename,
            "Original Cleaned Sentence": sentence
        })

encoding_df = pd.DataFrame(encoding_samples)

display(encoding_df)

,Document,Original Cleaned Sentence
0,attention_is_all_you_need.pdf,Abstract The dominant sequence transduction mo...
1,attention_is_all_you_need.pdf,The best performing models also connect the en...
2,attention_is_all_you_need.pdf,"We propose a new simple network architecture, ..."
3,bert.pdf,Abstract We introduce a new language represent...
4,bert.pdf,Unlike recent language repre- sentation models...
5,bert.pdf,"As a re- sult, the pre-trained BERT model can ..."
6,rag.pdf,Abstract Large pre-trained language models hav...
7,rag.pdf,"However, their ability to access and precisely..."
8,rag.pdf,"Additionally, providing provenance for their d..."


In [168]:
# Create separate corpora for classical encoding and sentence embeddings

processed_classical_corpus = []
embedding_corpus = []

processed_samples = []

for item in encoding_samples:
    original_sentence = item["Original Cleaned Sentence"]

    # For Bag of Words and TF-IDF:
    # apply normalization, stopword removal, and lemmatization
    normalized_sentence = normalize_text(original_sentence)
    stopword_removed_sentence = remove_stopwords(normalized_sentence)
    lemmatized_sentence = lemmatize_text(stopword_removed_sentence)

    processed_classical_corpus.append(lemmatized_sentence)

    # For sentence embeddings:
    # keep cleaned readable sentence
    embedding_corpus.append(original_sentence)

    processed_samples.append({
        "Document": item["Document"],
        "Original Cleaned Sentence": original_sentence,
        "Processed Sentence for BoW and TF-IDF": lemmatized_sentence
    })

processed_encoding_df = pd.DataFrame(processed_samples)

display(processed_encoding_df)

,Document,Original Cleaned Sentence,Processed Sentence for BoW and TF-IDF
0,attention_is_all_you_need.pdf,Abstract The dominant sequence transduction mo...,abstract dominant sequence transduction model ...
1,attention_is_all_you_need.pdf,The best performing models also connect the en...,good perform model also connect encoder decode...
2,attention_is_all_you_need.pdf,"We propose a new simple network architecture, ...",propose new simple network architecture transf...
3,bert.pdf,Abstract We introduce a new language represent...,abstract introduce new language representa tio...
4,bert.pdf,Unlike recent language repre- sentation models...,unlike recent language repre sentation model p...
5,bert.pdf,"As a re- sult, the pre-trained BERT model can ...",sult pre train bert model ne tune one addition...
6,rag.pdf,Abstract Large pre-trained language models hav...,abstract large pre train language model show s...
7,rag.pdf,"However, their ability to access and precisely...",however ability access precisely manipulate kn...
8,rag.pdf,"Additionally, providing provenance for their d...",additionally provide provenance decision updat...


In [169]:
# Check corpus sizes

print("Number of processed classical samples:", len(processed_classical_corpus))
print("Number of embedding samples:", len(embedding_corpus))

print("\nSample for BoW / TF-IDF:")
print(processed_classical_corpus[0])

print("\nSample for Sentence Embeddings:")
print(embedding_corpus[0])

Number of processed classical samples: 9
Number of embedding samples: 9

Sample for BoW / TF-IDF:
abstract dominant sequence transduction model base complex recurrent convolutional neural network include encoder decoder

Sample for Sentence Embeddings:
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.


## 13.2 Bag of Words

Bag of Words is a classical text encoding method that represents text using word counts.

It creates a vocabulary of unique words from the corpus. Each sentence is then represented as a vector showing how many times each word appears.

For this demonstration, the processed classical corpus is used because Bag of Words works better when the text has already been normalized, stopwords have been removed, and words have been lemmatized.

Bag of Words is simple and interpretable, but it ignores word order, grammar, and semantic meaning.

In [170]:
# Apply Bag of Words encoding

bow_vectorizer = CountVectorizer(max_features=25)

bow_matrix = bow_vectorizer.fit_transform(processed_classical_corpus)

bow_feature_names = bow_vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow_feature_names
)

display(bow_df)

,abstract,al,architecture,art,attention,base,bert,bidirectional,decoder,encoder,...,model,ne,network,new,pre,representation,state,task,train,tune
0,1,0,0,0,0,1,0,0,1,1,...,1,0,1,0,0,0,0,0,0,0
1,0,0,0,0,1,0,0,0,1,1,...,1,0,0,0,0,0,0,0,0,0
2,0,0,1,0,1,1,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
3,1,0,0,0,0,0,1,1,0,1,...,1,0,0,1,0,1,0,0,0,0
4,0,2,0,0,0,0,1,1,0,0,...,1,0,0,0,1,1,0,0,1,0
5,0,0,1,1,0,0,1,0,0,0,...,2,1,0,0,1,0,1,2,1,1
6,1,0,0,1,0,0,0,0,0,0,...,1,1,0,0,1,0,1,1,1,1
7,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,2,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [171]:
# Display Bag of Words vocabulary

print("Bag of Words Vocabulary:")
print(bow_feature_names)

Bag of Words Vocabulary:
['abstract' 'al' 'architecture' 'art' 'attention' 'base' 'bert'
 'bidirectional' 'decoder' 'encoder' 'et' 'knowledge' 'language' 'layer'
 'mechanism' 'model' 'ne' 'network' 'new' 'pre' 'representation' 'state'
 'task' 'train' 'tune']


## 13.3 Bag of Words Discussion

Bag of Words converts text into numerical vectors by counting word frequency. This makes text usable for machine learning and similarity-based comparison.

The advantage of Bag of Words is that it is simple, interpretable, and easy to understand. However, it has important limitations. It does not understand word order, grammar, or context. For example, two sentences with similar words but different meanings may receive similar representations.

In this project, Bag of Words is demonstrated as a classical NLP encoding method. However, it is not used as the final retrieval method because academic and technical question answering requires more contextual understanding.

## 13.4 TF-IDF

TF-IDF stands for Term Frequency-Inverse Document Frequency.

It improves on Bag of Words by considering not only how often a word appears, but also how important that word is across the whole corpus.

A word receives a higher TF-IDF score if it appears frequently in one sentence but does not appear frequently across all sentences. This helps highlight distinctive technical terms.

For this demonstration, the processed classical corpus is used because TF-IDF works better when text is normalized and lemmatized.

In [172]:
# Apply TF-IDF encoding

tfidf_vectorizer_demo = TfidfVectorizer(max_features=25)

tfidf_matrix_demo = tfidf_vectorizer_demo.fit_transform(processed_classical_corpus)

tfidf_feature_names = tfidf_vectorizer_demo.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix_demo.toarray(),
    columns=tfidf_feature_names
)

display(tfidf_df)

,abstract,al,architecture,art,attention,base,bert,bidirectional,decoder,encoder,...,model,ne,network,new,pre,representation,state,task,train,tune
0,0.393153,0.000000,0.000000,0.000000,0.000000,0.452175,0.000000,0.000000,0.452175,0.393153,...,0.278340,0.000000,0.452175,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.491777,0.000000,0.000000,0.000000,0.491777,0.427586,...,0.302717,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.362406,0.000000,0.416812,0.416812,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.416812,0.416812,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.348150,0.000000,0.000000,0.000000,0.000000,0.000000,0.348150,0.400416,0.000000,0.348150,...,0.246480,0.000000,0.000000,0.400416,0.000000,0.400416,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.566835,0.000000,0.000000,0.000000,0.000000,0.208133,0.239379,0.000000,0.000000,...,0.147352,0.000000,0.000000,0.000000,0.208133,0.239379,0.000000,0.000000,0.208133,0.000000
5,0.000000,0.000000,0.239735,0.275725,0.000000,0.000000,0.239735,0.000000,0.000000,0.000000,...,0.339450,0.275725,0.000000,0.000000,0.239735,0.000000,0.275725,0.479470,0.239735,0.275725
6,0.293952,0.000000,0.000000,0.338081,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.208109,0.338081,0.000000,0.000000,0.293952,0.000000,0.338081,0.293952,0.293952,0.338081
7,0.000000,0.000000,0.408248,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.816497,0.000000,0.000000
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [173]:
# Show top TF-IDF terms for each sample sentence

top_terms_data = []

for row_index in range(tfidf_matrix_demo.shape[0]):
    row = tfidf_matrix_demo[row_index].toarray().flatten()

    top_indices = row.argsort()[-5:][::-1]

    top_terms = [
        f"{tfidf_feature_names[i]} ({row[i]:.3f})"
        for i in top_indices
        if row[i] > 0
    ]

    top_terms_data.append({
        "Sentence Number": row_index + 1,
        "Top TF-IDF Terms": ", ".join(top_terms)
    })

top_terms_df = pd.DataFrame(top_terms_data)

display(top_terms_df)

,Sentence Number,Top TF-IDF Terms
0,1,"network (0.452), base (0.452), decoder (0.452)..."
1,2,"mechanism (0.492), decoder (0.492), attention ..."
2,3,"network (0.417), new (0.417), base (0.417), at..."
3,4,"representation (0.400), new (0.400), bidirecti..."
4,5,"et (0.567), al (0.567), bidirectional (0.239),..."
5,6,"task (0.479), model (0.339), tune (0.276), ne ..."
6,7,"tune (0.338), state (0.338), ne (0.338), art (..."
7,8,"task (0.816), architecture (0.408), knowledge ..."
8,9,knowledge (1.000)


## 13.5 TF-IDF Discussion

TF-IDF is more informative than Bag of Words because it gives importance to terms that are distinctive in a sentence or document.

For academic and technical documents, TF-IDF can highlight important terms such as "attention", "transformer", "retrieval", "generation", "encoder", and "model". This makes TF-IDF useful for building a classical retrieval baseline.

However, TF-IDF is still based on word overlap. If a user asks a question using different wording from the document, TF-IDF may fail to retrieve the best passage. This limitation motivates the use of sentence embeddings and RAG in later phases.

## 13.6 Sentence Embeddings

Sentence embeddings represent sentences as dense numerical vectors.

Unlike Bag of Words and TF-IDF, embeddings aim to capture semantic meaning. This means that two sentences can be considered similar even if they do not use exactly the same words.

For example, the question "What is the role of attention?" may be semantically related to a sentence discussing "attention mechanisms", even if the wording is not exactly identical.

For sentence embeddings, cleaned readable text is used because embeddings rely on sentence meaning, context, and natural language structure.

In [174]:
# Import sentence embedding model

from sentence_transformers import SentenceTransformer

print("SentenceTransformer imported successfully.")

SentenceTransformer imported successfully.


In [175]:
# Load a lightweight sentence embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence embedding model loaded successfully.


In [176]:
# Generate sentence embeddings for the cleaned readable sentences

sentence_embeddings = embedding_model.encode(embedding_corpus)

print("Embedding matrix shape:", sentence_embeddings.shape)

Embedding matrix shape: (9, 384)


In [177]:
# Display first 10 values of the first sentence embedding

print("Original sentence:")
print(embedding_corpus[0])

print("\nFirst 10 values of its embedding vector:")
print(sentence_embeddings[0][:10])

Original sentence:
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.

First 10 values of its embedding vector:
[-0.08288042 -0.12368692  0.07579745 -0.02420544  0.00629336  0.03695267
 -0.11137204  0.0117049   0.04853451 -0.03282288]


In [178]:
# Compare semantic similarity between sample sentences using embeddings

embedding_similarity_matrix = cosine_similarity(sentence_embeddings)

embedding_similarity_df = pd.DataFrame(
    embedding_similarity_matrix,
    columns=[f"Sentence {i+1}" for i in range(len(embedding_corpus))],
    index=[f"Sentence {i+1}" for i in range(len(embedding_corpus))]
)

display(embedding_similarity_df)

,Sentence 1,Sentence 2,Sentence 3,Sentence 4,Sentence 5,Sentence 6,Sentence 7,Sentence 8,Sentence 9
Sentence 1,1.000000,0.517075,0.346046,0.468089,0.424058,0.346430,0.338197,0.138827,0.038968
Sentence 2,0.517075,1.000000,0.524923,0.527670,0.457171,0.489341,0.294951,0.349622,0.039193
Sentence 3,0.346046,0.524923,1.000000,0.445241,0.356748,0.320312,0.177574,0.216860,-0.059448
Sentence 4,0.468089,0.527670,0.445241,1.000000,0.669469,0.569374,0.373813,0.176632,-0.059829
Sentence 5,0.424058,0.457171,0.356748,0.669469,1.000000,0.718399,0.563972,0.138507,0.020308
Sentence 6,0.346430,0.489341,0.320312,0.569374,0.718399,1.000000,0.591324,0.349628,0.031965
Sentence 7,0.338197,0.294951,0.177574,0.373813,0.563972,0.591324,1.000000,0.210344,0.151790
Sentence 8,0.138827,0.349622,0.216860,0.176632,0.138507,0.349628,0.210344,1.000000,0.241183
Sentence 9,0.038968,0.039193,-0.059448,-0.059829,0.020308,0.031965,0.151790,0.241183,1.000000


In [179]:
# Test semantic similarity between a user question and sample sentences

sample_question = "What is the Transformer model based on?"

question_embedding = embedding_model.encode([sample_question])

similarity_scores = cosine_similarity(question_embedding, sentence_embeddings).flatten()

similarity_results = []

for i, score in enumerate(similarity_scores):
    similarity_results.append({
        "Sentence Number": i + 1,
        "Similarity Score": round(score, 4),
        "Sentence": embedding_corpus[i]
    })

similarity_results_df = pd.DataFrame(similarity_results)

similarity_results_df = similarity_results_df.sort_values(
    by="Similarity Score",
    ascending=False
)

display(similarity_results_df)

,Sentence Number,Similarity Score,Sentence
3,4,0.4647,Abstract We introduce a new language represent...
2,3,0.4326,"We propose a new simple network architecture, ..."
1,2,0.3374,The best performing models also connect the en...
5,6,0.1926,"As a re- sult, the pre-trained BERT model can ..."
0,1,0.1899,Abstract The dominant sequence transduction mo...
4,5,0.1280,Unlike recent language repre- sentation models...
6,7,0.1110,Abstract Large pre-trained language models hav...
7,8,0.0645,"However, their ability to access and precisely..."
8,9,-0.0127,"Additionally, providing provenance for their d..."


The semantic similarity results show that sentence embeddings retrieve sentences based on meaning rather than exact keyword matching. For the question "What is the Transformer model based on?", the highest-ranked sentence discusses BERT because it contains related terms such as "language representation model" and "Transformers". However, the sentence that directly explains that the Transformer is based solely on attention mechanisms was still retrieved with a high similarity score. This shows both the strength and limitation of embeddings: they can capture semantic relationships, but they may sometimes rank a related sentence above the most directly relevant sentence. Therefore, retrieval systems usually return the top-k results rather than relying only on the top-1 result.

## 13.7 Comparison of Encoding Techniques

The three encoding techniques demonstrated in this phase have different strengths and limitations.

**Bag of Words** is simple and interpretable. It represents text using word counts, but it ignores word order and semantic meaning.

**TF-IDF** improves on Bag of Words by giving more weight to important and distinctive terms. It is useful for keyword-based retrieval, but it still depends heavily on exact word overlap.

**Sentence embeddings** represent text as dense semantic vectors. They are better at capturing meaning and can retrieve relevant text even when the question uses different wording from the document.

In this project, TF-IDF will be used as the classical retrieval baseline, while sentence embeddings will be used for semantic retrieval. These retrieval methods will later be compared with RAG-based question answering.

In [180]:
# Create a summary table comparing encoding techniques

encoding_summary = pd.DataFrame([
    {
        "Encoding Technique": "Bag of Words",
        "Text Version Used": "Processed lemmatized sample text",
        "Representation": "Sparse word-count vector",
        "Strength": "Simple and interpretable",
        "Limitation": "Ignores word order and meaning",
        "Use in This Project": "Demonstration of classical NLP encoding"
    },
    {
        "Encoding Technique": "TF-IDF",
        "Text Version Used": "Processed lemmatized sample text",
        "Representation": "Sparse weighted word vector",
        "Strength": "Highlights important terms",
        "Limitation": "Depends on exact word overlap",
        "Use in This Project": "Baseline retrieval system"
    },
    {
        "Encoding Technique": "Sentence Embeddings",
        "Text Version Used": "Cleaned readable sample text",
        "Representation": "Dense semantic vector",
        "Strength": "Captures semantic similarity",
        "Limitation": "Less directly interpretable than TF-IDF",
        "Use in This Project": "Semantic retrieval and RAG pipeline"
    }
])

display(encoding_summary)

,Encoding Technique,Text Version Used,Representation,Strength,Limitation,Use in This Project
0,Bag of Words,Processed lemmatized sample text,Sparse word-count vector,Simple and interpretable,Ignores word order and meaning,Demonstration of classical NLP encoding
1,TF-IDF,Processed lemmatized sample text,Sparse weighted word vector,Highlights important terms,Depends on exact word overlap,Baseline retrieval system
2,Sentence Embeddings,Cleaned readable sample text,Dense semantic vector,Captures semantic similarity,Less directly interpretable than TF-IDF,Semantic retrieval and RAG pipeline


## 13.8 Why Encoding Is Demonstrated on Samples First

Encoding techniques are first demonstrated on a small sample of sentences to keep the output readable and explainable.

Applying Bag of Words, TF-IDF, or embeddings directly to full research papers would produce very large matrices and vectors that are difficult to interpret in a notebook.

Therefore, this phase focuses on showing how each encoding method works. In the later retrieval phases, the full document collection will be split into chunks, and TF-IDF and sentence embeddings will be applied to all chunks for actual document retrieval and question answering.

## 14. Document Chunking

Document chunking is the process of splitting long documents into smaller text sections called chunks.

Academic and technical documents are usually too long to retrieve or pass directly into a question-answering model. Instead of searching across the entire document at once, the system searches across smaller chunks.

Chunking is important because:

1. It makes retrieval more precise.
2. It allows the system to find the most relevant parts of a document.
3. It helps RAG systems provide grounded answers using only selected context.
4. It avoids passing unnecessary document content to the language model.

In this project, chunking is applied to the cleaned document text because cleaned text preserves readability, sentence structure, and technical meaning.

## 14.1 Why Cleaned Text Is Used for Chunking

Chunking is performed on `cleaned_texts` rather than heavily normalized text.

The reason is that chunks will later be displayed to the user and passed into the RAG system as context. Therefore, the chunks should remain readable and preserve sentence structure, punctuation, and technical meaning.

Normalized or lemmatized text is useful for classical NLP processing, but it is less suitable as final context for question answering because it removes some readability and grammatical structure.

In [181]:
#Chunking Function
def create_sentence_chunks(text, chunk_size=5, overlap=1):
    """
    Splits text into sentence-based chunks with overlap.

    Parameters:
    text (str): Cleaned document text.
    chunk_size (int): Number of sentences per chunk.
    overlap (int): Number of sentences repeated between consecutive chunks.

    Returns:
    list: List of text chunks.
    """

    sentences = sent_tokenize(text)

    chunks = []
    start = 0

    while start < len(sentences):
        end = start + chunk_size
        chunk_sentences = sentences[start:end]

        chunk_text = " ".join(chunk_sentences).strip()

        if chunk_text:
            chunks.append(chunk_text)

        # Move forward while keeping overlap
        start += chunk_size - overlap

    return chunks

In [182]:
# Apply chunking to all cleaned documents

all_chunks = []

chunk_size = 5
overlap = 1

for filename, text in cleaned_texts.items():
    chunks = create_sentence_chunks(
        text=text,
        chunk_size=chunk_size,
        overlap=overlap
    )

    for i, chunk in enumerate(chunks, start=1):
        all_chunks.append({
            "Chunk ID": f"{filename}_chunk_{i}",
            "File Name": filename,
            "Chunk Number": i,
            "Chunk Text": chunk,
            "Word Count": len(chunk.split())
        })

chunks_df = pd.DataFrame(all_chunks)

print("Total number of chunks created:", len(chunks_df))
display(chunks_df.head())

Total number of chunks created: 180


,Chunk ID,File Name,Chunk Number,Chunk Text,Word Count
0,attention_is_all_you_need.pdf_chunk_1,attention_is_all_you_need.pdf,1,Abstract The dominant sequence transduction mo...,106
1,attention_is_all_you_need.pdf_chunk_2,attention_is_all_you_need.pdf,2,Our model achieves 28.4 BLEU on the WMT 2014 E...,96
2,attention_is_all_you_need.pdf_chunk_3,attention_is_all_you_need.pdf,3,Listing order is random. Jakob proposed replac...,76
3,attention_is_all_you_need.pdf_chunk_4,attention_is_all_you_need.pdf,4,"Niki designed, implemented, tuned and evaluate...",71
4,attention_is_all_you_need.pdf_chunk_5,attention_is_all_you_need.pdf,5,‡Work performed while at Google Research. 31st...,100


In [183]:
# Create chunking summary by document

chunking_summary_df = chunks_df.groupby("File Name").agg(
    Number_of_Chunks=("Chunk ID", "count"),
    Average_Word_Count=("Word Count", "mean"),
    Minimum_Word_Count=("Word Count", "min"),
    Maximum_Word_Count=("Word Count", "max")
).reset_index()

chunking_summary_df["Average_Word_Count"] = chunking_summary_df["Average_Word_Count"].round(2)

display(chunking_summary_df)

,File Name,Number_of_Chunks,Average_Word_Count,Minimum_Word_Count,Maximum_Word_Count
0,attention_is_all_you_need.pdf,53,107.17,17,234
1,bert.pdf,63,122.41,29,209
2,rag.pdf,64,109.39,8,213


In [184]:
# Display sample chunks

sample_chunks = chunks_df.head(3)

for index, row in sample_chunks.iterrows():
    print("Chunk ID:", row["Chunk ID"])
    print("File Name:", row["File Name"])
    print("Chunk Number:", row["Chunk Number"])
    print("Word Count:", row["Word Count"])
    print("-" * 80)
    print(row["Chunk Text"])
    print("=" * 100)

Chunk ID: attention_is_all_you_need.pdf_chunk_1
File Name: attention_is_all_you_need.pdf
Chunk Number: 1
Word Count: 106
--------------------------------------------------------------------------------
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU.
Chunk ID: attention_is_all_you_need.pdf_chunk_2
File Name: atte

In [185]:
# Basic statistics for chunk word counts

chunks_df["Word Count"].describe()

,Word Count
count,180.000000
mean,113.294444
std,35.780163
min,8.000000
25%,90.000000
50%,107.500000
75%,136.250000
max,234.000000


In [186]:
# Save chunks to CSV for later retrieval phases

chunks_output_path = "outputs/retrieved_chunks/document_chunks.csv"

chunks_df.to_csv(chunks_output_path, index=False)

print("Document chunks saved successfully at:", chunks_output_path)

Document chunks saved successfully at: outputs/retrieved_chunks/document_chunks.csv


In [187]:
# Verify saved chunk file

if os.path.exists(chunks_output_path):
    file_size_kb = os.path.getsize(chunks_output_path) / 1024
    print(f"Chunk file exists: {chunks_output_path}")
    print(f"File size: {file_size_kb:.2f} KB")
else:
    print("Chunk file was not found.")

Chunk file exists: outputs/retrieved_chunks/document_chunks.csv
File size: 135.43 KB


## 14.2 Chunking Discussion

The documents were split into sentence-based chunks instead of fixed character-based chunks. This helps preserve readability because each chunk contains complete sentences.

In this project, each chunk is not a single sentence. Instead, each chunk contains a group of five sentences, with one sentence overlapping between consecutive chunks. This approach provides enough context for retrieval and question answering while keeping each chunk small enough to search efficiently.

An overlap of one sentence was used between consecutive chunks. This is useful because important information may continue across sentence boundaries. Overlap helps preserve context between neighbouring chunks.

The chunks created in this phase will be used in the next stages for:

1. TF-IDF based retrieval
2. Embedding-based semantic retrieval
3. RAG-based question answering

This step connects preprocessing with the final document question-answering system.

## 15. TF-IDF Baseline Retrieval System

In this phase, a classical TF-IDF based retrieval system is built.

TF-IDF retrieval works by converting both the document chunks and the user question into numerical vectors. The similarity between the question vector and each chunk vector is then calculated using cosine similarity.

The chunks with the highest similarity scores are returned as the most relevant results.

This is used as the baseline retrieval system because TF-IDF is a classical NLP technique that is simple, interpretable, and effective for keyword-based search.

However, TF-IDF depends heavily on exact word overlap. If the question uses different wording from the document, TF-IDF may not always retrieve the most relevant chunk. This limitation will be compared later with embedding-based semantic retrieval.

In [188]:
# Prepare chunk text for TF-IDF retrieval

def preprocess_for_tfidf(text):
    """
    Preprocesses text for TF-IDF retrieval.

    Steps:
    1. Normalize text
    2. Remove stopwords
    3. Lemmatize text using spaCy

    Parameters:
    text (str): Input text.

    Returns:
    str: Preprocessed text.
    """

    normalized = normalize_text(text)
    no_stopwords = remove_stopwords(normalized)
    lemmatized = lemmatize_text(no_stopwords)

    return lemmatized

In [189]:
# Apply preprocessing to all document chunks

chunks_df["Processed Chunk Text"] = chunks_df["Chunk Text"].apply(preprocess_for_tfidf)

display(chunks_df[["Chunk ID", "File Name", "Chunk Number", "Chunk Text", "Processed Chunk Text"]].head())

,Chunk ID,File Name,Chunk Number,Chunk Text,Processed Chunk Text
0,attention_is_all_you_need.pdf_chunk_1,attention_is_all_you_need.pdf,1,Abstract The dominant sequence transduction mo...,abstract dominant sequence transduction model ...
1,attention_is_all_you_need.pdf_chunk_2,attention_is_all_you_need.pdf,2,Our model achieves 28.4 BLEU on the WMT 2014 E...,model achieve 28.4 bleu wmt 2014 english germa...
2,attention_is_all_you_need.pdf_chunk_3,attention_is_all_you_need.pdf,3,Listing order is random. Jakob proposed replac...,list order random jakob propose replace rnns s...
3,attention_is_all_you_need.pdf_chunk_4,attention_is_all_you_need.pdf,4,"Niki designed, implemented, tuned and evaluate...",niki design implement tune evaluate countless ...
4,attention_is_all_you_need.pdf_chunk_5,attention_is_all_you_need.pdf,5,‡Work performed while at Google Research. 31st...,work perform google research 31st conference n...


In [190]:
# Create TF-IDF vectors for all processed chunks

tfidf_vectorizer = TfidfVectorizer()

tfidf_chunk_matrix = tfidf_vectorizer.fit_transform(chunks_df["Processed Chunk Text"])

print("TF-IDF matrix shape:", tfidf_chunk_matrix.shape)

TF-IDF matrix shape: (180, 2104)


In [191]:
#TF-IDF Retrieval Function
def retrieve_tfidf(query, top_k=3):
    """
    Retrieves the top-k most relevant chunks using TF-IDF and cosine similarity.

    Parameters:
    query (str): User question.
    top_k (int): Number of relevant chunks to retrieve.

    Returns:
    pandas.DataFrame: Top-k retrieved chunks with similarity scores.
    """

    # Apply the same preprocessing to the query
    processed_query = preprocess_for_tfidf(query)

    # Convert query into TF-IDF vector
    query_vector = tfidf_vectorizer.transform([processed_query])

    # Calculate cosine similarity between query and all chunks
    similarity_scores = cosine_similarity(query_vector, tfidf_chunk_matrix).flatten()

    # Get top-k chunk indices
    top_indices = similarity_scores.argsort()[-top_k:][::-1]

    # Prepare results
    results = chunks_df.iloc[top_indices].copy()
    results["Similarity Score"] = similarity_scores[top_indices]

    # Keep useful columns
    results = results[
        [
            "Chunk ID",
            "File Name",
            "Chunk Number",
            "Similarity Score",
            "Chunk Text"
        ]
    ]

    return results

In [192]:
# Test the TF-IDF retrieval system

sample_question = "What is the Transformer model based on?"

tfidf_results = retrieve_tfidf(sample_question, top_k=3)

display(tfidf_results)

,Chunk ID,File Name,Chunk Number,Similarity Score,Chunk Text
50,attention_is_all_you_need.pdf_chunk_51,attention_is_all_you_need.pdf,51,0.301091,"Conclusion In this work, we presented the Tran..."
39,attention_is_all_you_need.pdf_chunk_40,attention_is_all_you_need.pdf,40,0.248526,Even our base model surpasses all previously p...
38,attention_is_all_you_need.pdf_chunk_39,attention_is_all_you_need.pdf,39,0.247454,"This hurts perplexity, as the model learns to ..."


In [193]:
# Display retrieved chunks clearly

print("Question:", sample_question)
print("=" * 100)

for index, row in tfidf_results.iterrows():
    print("Chunk ID:", row["Chunk ID"])
    print("File Name:", row["File Name"])
    print("Chunk Number:", row["Chunk Number"])
    print("Similarity Score:", round(row["Similarity Score"], 4))
    print("-" * 100)
    print(row["Chunk Text"])
    print("=" * 100)

Question: What is the Transformer model based on?
Chunk ID: attention_is_all_you_need.pdf_chunk_51
File Name: attention_is_all_you_need.pdf
Chunk Number: 51
Similarity Score: 0.3011
----------------------------------------------------------------------------------------------------
Conclusion In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014 English-to-French translation tasks, we achieve a new state of the art. In the former task our best model outperforms even all previously reported ensembles. We are excited about the future of attention-based models and plan to apply them to other tasks.
Chunk ID: attention_is_all_you_need.p

In [194]:
# Test TF-IDF retrieval with multiple sample questions

test_questions = [
    "What is the Transformer model based on?",
    "What does BERT stand for?",
    "What is Retrieval-Augmented Generation?",
    "Why is attention useful in sequence transduction?",
    "How does RAG use retrieved documents?"
]

for question in test_questions:
    print("Question:", question)
    print("=" * 100)

    results = retrieve_tfidf(question, top_k=2)

    for index, row in results.iterrows():
        print("File Name:", row["File Name"])
        print("Chunk Number:", row["Chunk Number"])
        print("Similarity Score:", round(row["Similarity Score"], 4))
        print("Chunk Preview:", row["Chunk Text"][:500])
        print("-" * 100)

    print("\n")

Question: What is the Transformer model based on?
File Name: attention_is_all_you_need.pdf
Chunk Number: 51
Similarity Score: 0.3011
Chunk Preview: Conclusion In this work, we presented the Transformer, the first sequence transduction model based entirely on attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with multi-headed self-attention. For translation tasks, the Transformer can be trained significantly faster than architectures based on recurrent or convolutional layers. On both WMT 2014 English-to-German and WMT 2014 English-to-French translation tasks, we achieve a new state of the art. In 
----------------------------------------------------------------------------------------------------
File Name: attention_is_all_you_need.pdf
Chunk Number: 40
Similarity Score: 0.2485
Chunk Preview: Even our base model surpasses all previously published models and ensembles, at a fraction of the training cost of any of the competitive models. On the

The TF-IDF baseline retrieval system successfully retrieved relevant chunks for most sample questions. For example, the BERT question retrieved the chunk that directly states that BERT stands for Bidirectional Encoder Representations from Transformers. The Transformer question also retrieved a relevant chunk explaining that the Transformer is based entirely on attention.

The results improved after removing the References section from the cleaned text, because bibliography-related chunks were no longer ranked highly due to keyword overlap.

However, the results also show a limitation of TF-IDF. For the question about how RAG uses retrieved documents, TF-IDF retrieved chunks that contain related keywords such as "retrieved documents" and "RAG", but the retrieved chunks focus more on the effect of retrieving more documents rather than directly explaining the mechanism. This shows that TF-IDF is useful and interpretable, but it depends heavily on word overlap and may not always capture deeper semantic meaning.

In [195]:
# Create a compact results table for multiple questions

tfidf_evaluation_rows = []

for question in test_questions:
    results = retrieve_tfidf(question, top_k=1)

    top_result = results.iloc[0]

    tfidf_evaluation_rows.append({
        "Question": question,
        "Top Retrieved File": top_result["File Name"],
        "Top Retrieved Chunk Number": top_result["Chunk Number"],
        "Similarity Score": round(top_result["Similarity Score"], 4),
        "Top Retrieved Chunk Preview": top_result["Chunk Text"][:300]
    })

tfidf_evaluation_df = pd.DataFrame(tfidf_evaluation_rows)

display(tfidf_evaluation_df)

,Question,Top Retrieved File,Top Retrieved Chunk Number,Similarity Score,Top Retrieved Chunk Preview
0,What is the Transformer model based on?,attention_is_all_you_need.pdf,51,0.3011,"Conclusion In this work, we presented the Tran..."
1,What does BERT stand for?,bert.pdf,1,0.1816,Abstract We introduce a new language represent...
2,What is Retrieval-Augmented Generation?,rag.pdf,6,0.2136,"Here, we bring hybrid parametric and non-param..."
3,Why is attention useful in sequence transduction?,attention_is_all_you_need.pdf,9,0.3638,"Self-attention, sometimes called intra-attenti..."
4,How does RAG use retrieved documents?,rag.pdf,54,0.5231,Figure 3 (left) shows that retrieving more doc...


## 15.1 TF-IDF Retrieval Discussion

The TF-IDF baseline retrieval system converts both user questions and document chunks into weighted keyword vectors. Cosine similarity is then used to identify the chunks that are most similar to the question.

This method is useful because it is simple, explainable, and based on classical NLP concepts. The similarity scores show why certain chunks were retrieved.

TF-IDF performs well when the question contains keywords that also appear in the document. For example, a question about "Transformer", "attention", or "BERT" is likely to retrieve relevant chunks if those terms appear in the text.

However, TF-IDF has limitations. It depends heavily on word overlap and may struggle when the question uses different wording from the document. It also does not deeply understand semantic meaning. These limitations motivate the next phase, where sentence embeddings will be used for semantic retrieval.

## 16. Embedding-Based Retrieval with FAISS

In this phase, an embedding-based retrieval system is built.

Unlike TF-IDF, which relies heavily on exact word overlap, embedding-based retrieval represents both document chunks and user questions as dense semantic vectors. These vectors capture the meaning of the text.

If a question and a document chunk have similar meaning, their embeddings will be close to each other in vector space, even if they do not use exactly the same words.

FAISS is used to store and search the chunk embeddings efficiently. This allows the system to retrieve the most semantically relevant chunks for a user question.

This retrieval method will later support the RAG-based question-answering system.

In [196]:
# Import libraries for embedding-based retrieval

from sentence_transformers import SentenceTransformer
import faiss

print("Embedding retrieval libraries imported successfully.")

Embedding retrieval libraries imported successfully.


In [197]:
# Load sentence embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Sentence embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence embedding model loaded successfully.


In [198]:
# Prepare readable chunk texts for embedding-based retrieval

chunk_texts = chunks_df["Chunk Text"].tolist()
#We use Chunk Text, not Processed Chunk Text, because sentence embeddings work better with readable natural language.

print("Number of chunks to embed:", len(chunk_texts))
print("\nSample chunk:")
print(chunk_texts[0][:1000])

Number of chunks to embed: 180

Sample chunk:
Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English- to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU.


In [199]:
# Generate embeddings for all document chunks

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)

print("Chunk embedding shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Chunk embedding shape: (180, 384)


In [211]:
# Display a small part of the chunk embedding matrix

embedding_matrix_df = pd.DataFrame(
    chunk_embeddings[:5, :10],
    columns=[f"Dimension_{i+1}" for i in range(10)]
)

embedding_matrix_df.insert(0, "Chunk ID", chunks_df["Chunk ID"].head(5).values)

display(embedding_matrix_df)

,Chunk ID,Dimension_1,Dimension_2,Dimension_3,Dimension_4,Dimension_5,Dimension_6,Dimension_7,Dimension_8,Dimension_9,Dimension_10
0,attention_is_all_you_need.pdf_chunk_1,-0.064570,-0.106705,0.040092,-0.010109,0.017417,0.026790,-0.086282,0.036608,0.093970,-0.062416
1,attention_is_all_you_need.pdf_chunk_2,-0.052261,-0.081527,0.039216,0.019294,0.023263,0.035966,-0.041128,0.042209,0.041813,-0.022794
2,attention_is_all_you_need.pdf_chunk_3,-0.103475,-0.082654,-0.018539,-0.043175,-0.008786,0.013798,-0.020935,0.063975,0.025158,-0.023548
3,attention_is_all_you_need.pdf_chunk_4,-0.065317,-0.106163,-0.005060,-0.033041,0.021741,-0.014538,-0.115308,0.016377,-0.065118,-0.061746
4,attention_is_all_you_need.pdf_chunk_5,-0.069988,-0.095566,0.095082,-0.032374,0.005687,0.050571,-0.029453,-0.007324,0.043176,-0.061219


In [200]:
# Convert embeddings to float32 because FAISS requires float32 vectors

chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print("Embedding data type:", chunk_embeddings.dtype)
print("Embedding shape:", chunk_embeddings.shape)

Embedding data type: float32
Embedding shape: (180, 384)


In [202]:
# Normalize embeddings so inner product behaves like cosine similarity

faiss.normalize_L2(chunk_embeddings)

print("Chunk embeddings normalized successfully.")

#FAISS can use inner product search. After normalization, inner product similarity is equivalent to cosine similarity.

Chunk embeddings normalized successfully.


In [203]:
# Create FAISS index for semantic search

embedding_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(chunk_embeddings)

print("FAISS index created successfully.")
print("Number of vectors in FAISS index:", faiss_index.ntotal)

FAISS index created successfully.
Number of vectors in FAISS index: 180


In [204]:
#Embedding Retrieval Function
def retrieve_embeddings(query, top_k=3):
    """
    Retrieves the top-k most semantically relevant chunks using sentence embeddings and FAISS.

    Parameters:
    query (str): User question.
    top_k (int): Number of relevant chunks to retrieve.

    Returns:
    pandas.DataFrame: Top-k retrieved chunks with similarity scores.
    """

    # Convert query into embedding
    query_embedding = embedding_model.encode([query])

    # Convert to float32
    query_embedding = np.array(query_embedding).astype("float32")

    # Normalize query embedding
    faiss.normalize_L2(query_embedding)

    # Search FAISS index
    similarity_scores, top_indices = faiss_index.search(query_embedding, top_k)

    # Prepare results
    results = chunks_df.iloc[top_indices[0]].copy()
    results["Similarity Score"] = similarity_scores[0]

    results = results[
        [
            "Chunk ID",
            "File Name",
            "Chunk Number",
            "Similarity Score",
            "Chunk Text"
        ]
    ]

    return results

In [205]:
# Test embedding-based retrieval

sample_question = "What is the Transformer model based on?"

embedding_results = retrieve_embeddings(sample_question, top_k=3)

display(embedding_results)

,Chunk ID,File Name,Chunk Number,Similarity Score,Chunk Text
10,attention_is_all_you_need.pdf_chunk_11,attention_is_all_you_need.pdf,11,0.505777,At each step the model is auto-regressive [10]...
69,bert.pdf_chunk_17,bert.pdf,17,0.470518,There is mini- mal difference between the pre-...
41,attention_is_all_you_need.pdf_chunk_42,attention_is_all_you_need.pdf,42,0.453620,Table 2 summarizes our results and compares ou...


In [206]:
# Display retrieved chunks clearly

print("Question:", sample_question)
print("=" * 100)

for index, row in embedding_results.iterrows():
    print("Chunk ID:", row["Chunk ID"])
    print("File Name:", row["File Name"])
    print("Chunk Number:", row["Chunk Number"])
    print("Similarity Score:", round(row["Similarity Score"], 4))
    print("-" * 100)
    print(row["Chunk Text"])
    print("=" * 100)

Question: What is the Transformer model based on?
Chunk ID: attention_is_all_you_need.pdf_chunk_11
File Name: attention_is_all_you_need.pdf
Chunk Number: 11
Similarity Score: 0.5058
----------------------------------------------------------------------------------------------------
At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next. Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers.
Chunk ID: bert.pdf_chunk_17
File Name: bert.pdf
Chunk Number: 17
Similarity Score: 0.4705
-----------------------------------------------------------------------------------------------

In [207]:
# Test embedding retrieval with multiple sample questions

test_questions = [
    "What is the Transformer model based on?",
    "What does BERT stand for?",
    "What is Retrieval-Augmented Generation?",
    "Why is attention useful in sequence transduction?",
    "How does RAG use retrieved documents?"
]

for question in test_questions:
    print("Question:", question)
    print("=" * 100)

    results = retrieve_embeddings(question, top_k=2)

    for index, row in results.iterrows():
        print("File Name:", row["File Name"])
        print("Chunk Number:", row["Chunk Number"])
        print("Similarity Score:", round(row["Similarity Score"], 4))
        print("Chunk Preview:", row["Chunk Text"][:500])
        print("-" * 100)

    print("\n")

Question: What is the Transformer model based on?
File Name: attention_is_all_you_need.pdf
Chunk Number: 11
Similarity Score: 0.5058
Chunk Preview: At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next. Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers.
----------------------------------------------------------------------------------------------------
File Name: bert.pdf
Chunk Number: 17
Similarity Score: 0.4705
Chunk Preview: There is mini- mal difference between the pre-trained architec- ture and the ﬁnal downstream architecture. Model Architecture BERT’s model architec- ture is a multi-layer bi

The embedding-based retrieval system retrieved semantically related chunks for the sample questions. It performed well for broader conceptual questions, such as why attention is useful in sequence transduction, because the retrieved chunks discussed related ideas such as self-attention, sequence representation, and parallelization.

However, the results also show that embedding retrieval does not always retrieve the most direct answer for exact-definition questions. For example, for the question "What does BERT stand for?", the embedding system retrieved chunks about BERT architecture and token sequences rather than the abstract chunk that directly defines BERT as Bidirectional Encoder Representations from Transformers.

This demonstrates an important difference between TF-IDF and embedding-based retrieval. TF-IDF can perform well when the question and document share exact keywords, while embedding retrieval is better at capturing broader semantic similarity. Therefore, embedding retrieval is useful for conceptual questions, but exact acronym or definition-based questions may sometimes be better handled by TF-IDF or by retrieving a larger top-k set.

These results show why retrieval systems often use top-k retrieval instead of relying only on the top result. In the final RAG system, multiple retrieved chunks can be passed as context so that the answer generator has access to more relevant information.

- TF-IDF is better for exact keyword and acronym-based questions.
- Embedding retrieval is better for semantic and conceptual questions.
- RAG can benefit from top-k retrieval by using multiple retrieved chunks as context.

In [208]:
# Create a compact results table for embedding retrieval

embedding_evaluation_rows = []

for question in test_questions:
    results = retrieve_embeddings(question, top_k=1)

    top_result = results.iloc[0]

    embedding_evaluation_rows.append({
        "Question": question,
        "Top Retrieved File": top_result["File Name"],
        "Top Retrieved Chunk Number": top_result["Chunk Number"],
        "Similarity Score": round(top_result["Similarity Score"], 4),
        "Top Retrieved Chunk Preview": top_result["Chunk Text"][:300]
    })

embedding_evaluation_df = pd.DataFrame(embedding_evaluation_rows)

display(embedding_evaluation_df)

,Question,Top Retrieved File,Top Retrieved Chunk Number,Similarity Score,Top Retrieved Chunk Preview
0,What is the Transformer model based on?,attention_is_all_you_need.pdf,11,0.5058,At each step the model is auto-regressive [10]...
1,What does BERT stand for?,bert.pdf,16,0.4972,"For ﬁne- tuning, the BERT model is ﬁrst initia..."
2,What is Retrieval-Augmented Generation?,rag.pdf,60,0.5833,"This said, RAG techniques may work well in the..."
3,Why is attention useful in sequence transduction?,attention_is_all_you_need.pdf,6,0.6253,Recurrent models typically factor computation ...
4,How does RAG use retrieved documents?,rag.pdf,54,0.6498,Figure 3 (left) shows that retrieving more doc...


In [209]:
# Compare top-1 TF-IDF retrieval and top-1 embedding retrieval

comparison_rows = []

for question in test_questions:
    tfidf_top = retrieve_tfidf(question, top_k=1).iloc[0]
    embedding_top = retrieve_embeddings(question, top_k=1).iloc[0]

    comparison_rows.append({
        "Question": question,
        "TF-IDF Top File": tfidf_top["File Name"],
        "TF-IDF Chunk": tfidf_top["Chunk Number"],
        "TF-IDF Score": round(tfidf_top["Similarity Score"], 4),
        "Embedding Top File": embedding_top["File Name"],
        "Embedding Chunk": embedding_top["Chunk Number"],
        "Embedding Score": round(embedding_top["Similarity Score"], 4)
    })

retrieval_comparison_df = pd.DataFrame(comparison_rows)

display(retrieval_comparison_df)

,Question,TF-IDF Top File,TF-IDF Chunk,TF-IDF Score,Embedding Top File,Embedding Chunk,Embedding Score
0,What is the Transformer model based on?,attention_is_all_you_need.pdf,51,0.3011,attention_is_all_you_need.pdf,11,0.5058
1,What does BERT stand for?,bert.pdf,1,0.1816,bert.pdf,16,0.4972
2,What is Retrieval-Augmented Generation?,rag.pdf,6,0.2136,rag.pdf,60,0.5833
3,Why is attention useful in sequence transduction?,attention_is_all_you_need.pdf,9,0.3638,attention_is_all_you_need.pdf,6,0.6253
4,How does RAG use retrieved documents?,rag.pdf,54,0.5231,rag.pdf,54,0.6498


In [212]:
# Display TF-IDF and embedding retrieval side by side for one question

comparison_question = "How does RAG combine parametric memory and non-parametric memory?"

tfidf_result = retrieve_tfidf(comparison_question, top_k=1).iloc[0]
embedding_result = retrieve_embeddings(comparison_question, top_k=1).iloc[0]

print("Question:", comparison_question)
print("=" * 100)

print("TF-IDF Top Result")
print("-" * 100)
print("File:", tfidf_result["File Name"])
print("Chunk Number:", tfidf_result["Chunk Number"])
print("Similarity Score:", round(tfidf_result["Similarity Score"], 4))
print(tfidf_result["Chunk Text"][:1000])

print("\n" + "=" * 100 + "\n")

print("Embedding-Based Top Result")
print("-" * 100)
print("File:", embedding_result["File Name"])
print("Chunk Number:", embedding_result["Chunk Number"])
print("Similarity Score:", round(embedding_result["Similarity Score"], 4))
print(embedding_result["Chunk Text"][:1000])

Question: How does RAG combine parametric memory and non-parametric memory?
TF-IDF Top Result
----------------------------------------------------------------------------------------------------
File: rag.pdf
Chunk Number: 6
Similarity Score: 0.6846
Here, we bring hybrid parametric and non-parametric memory to the “workhorse of NLP,” i.e. sequence-to-sequence (seq2seq) models. We endow pre-trained, parametric-memory generation models with a non-parametric memory through a general-purpose ﬁne-tuning approach which we refer to as retrieval-augmented generation (RAG). We build RAG models where the parametric memory is a pre-trained seq2seq transformer, and the non-parametric memory is a dense vector index of Wikipedia, accessed with a pre-trained neural retriever. We combine these components in a probabilistic model trained end-to-end (Fig.


Embedding-Based Top Result
----------------------------------------------------------------------------------------------------
File: rag.pdf
Chunk 

For the question "How does RAG combine parametric memory and non-parametric memory?", both TF-IDF and embedding-based retrieval returned relevant chunks. The TF-IDF result was more directly explanatory because it retrieved the chunk that explicitly states that RAG uses a pre-trained seq2seq Transformer as parametric memory and a dense vector index of Wikipedia as non-parametric memory. This occurred because the question shared exact terms with the chunk, such as "parametric memory" and "non-parametric memory".

The embedding-based result was also relevant, but it focused more broadly on the idea that RAG combines pre-trained parametric and non-parametric memory components and jointly learns the generator and retriever. This shows that TF-IDF can perform very well for exact technical terminology, while embedding retrieval can capture broader semantic relationships.

## 16.1 Embedding Retrieval Discussion

The embedding-based retrieval system uses sentence embeddings to represent document chunks and user questions as dense semantic vectors.

Unlike TF-IDF, embedding retrieval does not depend only on exact word overlap. It can retrieve relevant chunks even when the question uses different wording from the document.

For example, a question about how RAG uses retrieved documents may retrieve chunks discussing non-parametric memory, dense vector indexes, retrieved passages, or knowledge-intensive tasks, even if the exact question words are not repeated.

FAISS is used to store and search the embeddings efficiently. After normalizing the embeddings, inner product similarity is used as cosine similarity.

Embedding retrieval is more suitable than TF-IDF when semantic meaning is important. However, it is less directly interpretable than TF-IDF because the individual embedding dimensions do not have clear human-readable meanings.

## 16.2 TF-IDF vs Embedding Retrieval

TF-IDF and embedding-based retrieval have different strengths.

TF-IDF is simple, transparent, and useful when the user question shares keywords with the document chunk. Its similarity scores are easier to explain because they are based on weighted word overlap.

Embedding-based retrieval is better at capturing semantic similarity. It can identify relevant chunks even when the wording of the question differs from the wording in the document.

However, embedding retrieval is less interpretable because dense vector dimensions are not directly meaningful to humans.

In this project, TF-IDF serves as the classical NLP baseline, while embedding-based retrieval serves as the semantic retrieval method. The next phase will use retrieved chunks as context for Retrieval-Augmented Generation.

## 17. RAG-Based Question Answering System using Gemini API

Retrieval-Augmented Generation, or RAG, combines document retrieval with answer generation.

Instead of asking a language model to answer from memory, RAG first retrieves relevant chunks from the document collection. These retrieved chunks are then passed to the language model as context. The model generates an answer using only the retrieved context.

In this project, the retrieval pipeline is built manually using sentence embeddings and FAISS. Gemini API is used only for the final answer generation step.

The RAG pipeline follows these steps:

1. The user asks a question.
2. The question is converted into an embedding.
3. FAISS retrieves the most semantically relevant document chunks.
4. The retrieved chunks are combined into a context.
5. The context and question are passed to Gemini.
6. Gemini generates an answer based only on the retrieved document context.
7. The retrieved source chunks are displayed to make the answer explainable.

This makes the question-answering system more grounded and transparent.

In [217]:
# Install Google GenAI SDK

!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.4/790.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.5/246.5 kB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.50.0 which is incompatible.


In [232]:
# Import Gemini API tools

from google import genai
from google.colab import userdata

print("Gemini SDK imported successfully.")

Gemini SDK imported successfully.


In [233]:
# Create Gemini API client using API key stored in Colab Secrets

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if GEMINI_API_KEY is None:
    raise ValueError("Gemini API key not found. Please add GEMINI_API_KEY in Colab Secrets.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini API client created successfully.")

Gemini API client created successfully.


In [234]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one short sentence."
)

print(response.text)

Hi!


In [235]:
#Function to create Context from Retrieved Chunks
def build_rag_context(retrieved_chunks, max_chars_per_chunk=1200):
    """
    Builds a context string from retrieved chunks.

    Parameters:
    retrieved_chunks (pandas.DataFrame): Retrieved chunks from embedding retrieval.
    max_chars_per_chunk (int): Maximum characters to keep from each chunk.

    Returns:
    str: Combined context for RAG.
    """

    context_parts = []

    for _, row in retrieved_chunks.iterrows():
        source_info = (
            f"[Source: {row['File Name']}, "
            f"Chunk: {row['Chunk Number']}, "
            f"Score: {row['Similarity Score']:.4f}]"
        )

        chunk_text = row["Chunk Text"][:max_chars_per_chunk]

        context_parts.append(source_info + "\n" + chunk_text)

    context = "\n\n".join(context_parts)

    return context

In [236]:
#Gemini RAG Answer Function (Only using Embeddings)
def generate_rag_answer_gemini_embedding_only(question, top_k=3):
    """
    Generates an answer using Gemini API with retrieved document chunks.

    Steps:
    1. Retrieve relevant chunks using embedding retrieval.
    2. Build context from retrieved chunks.
    3. Send context and question to Gemini.
    4. Return generated answer, retrieved source chunks, and context.

    Parameters:
    question (str): User question.
    top_k (int): Number of chunks to retrieve.

    Returns:
    tuple:
        answer (str): Generated answer.
        retrieved_chunks (pandas.DataFrame): Retrieved chunks used as sources.
        context (str): Context sent to Gemini.
    """

    # Retrieve relevant chunks using embedding-based retrieval
    retrieved_chunks = retrieve_embeddings(question, top_k=top_k)

    # Build RAG context from retrieved chunks
    context = build_rag_context(
        retrieved_chunks,
        max_chars_per_chunk=1200
    )

    # Create grounded prompt
    prompt = f"""
You are an academic document question-answering assistant.

Answer the question using only the provided context.
Do not use outside knowledge.
If the answer is not available in the context, say:
"The document context does not provide enough information."

Keep the answer clear, concise, and academic.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer using Gemini
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt
    )

    answer = response.text

    return answer, retrieved_chunks, context

In [237]:
# Test Gemini-based RAG question answering

sample_question = "How does RAG combine parametric memory and non-parametric memory?"

rag_answer, rag_sources, rag_context = generate_rag_answer_gemini_embedding_only(
    question=sample_question,
    top_k=3
)

print("Question:")
print(sample_question)

print("\nGenerated Answer:")
print("=" * 100)
print(rag_answer)

Question:
How does RAG combine parametric memory and non-parametric memory?

Generated Answer:
RAG combines parametric and non-parametric memory by functioning as a hybrid model that integrates a pre-trained neural language model (parameterized implicit knowledge base) with a retrieval-based (non-parametric) memory. In this architecture, the non-parametric component guides the generation process by drawing out specific knowledge stored within the parametric memory. Both the generator and the retriever are jointly learned during fine-tuning on sequence-to-sequence (seq2seq) tasks.


In [238]:
# Display retrieved source chunks used for answer generation
#This makes the RAG system explainable because the answer can be checked against the retrieved chunks.

print("Retrieved Source Chunks Used for Answer:")
print("=" * 100)

for _, row in rag_sources.iterrows():
    print("Chunk ID:", row["Chunk ID"])
    print("File Name:", row["File Name"])
    print("Chunk Number:", row["Chunk Number"])
    print("Similarity Score:", round(row["Similarity Score"], 4))
    print("-" * 100)
    print(row["Chunk Text"][:1200])
    print("=" * 100)

Retrieved Source Chunks Used for Answer:
Chunk ID: rag.pdf_chunk_8
File Name: rag.pdf
Chunk Number: 8
Similarity Score: 0.567
----------------------------------------------------------------------------------------------------
Like T5 [51] or BART, RAG can be ﬁne-tuned on any seq2seq task, whereby both the generator and retriever are jointly learned. There has been extensive previous work proposing architectures to enrich systems with non-parametric memory which are trained from scratch for speciﬁc tasks, e.g. memory networks [64, 55], stack- augmented networks [25] and memory layers [30]. In contrast, we explore a setting where both parametric and non-parametric memory components are pre-trained and pre-loaded with extensive knowledge. Crucially, by using pre-trained access mechanisms, the ability to access knowledge is present without additional training.
Chunk ID: rag.pdf_chunk_42
File Name: rag.pdf
Chunk Number: 42
Similarity Score: 0.5637
------------------------------------------

In [239]:
# Display the full context passed to Gemini

print("Full RAG Context Sent to Gemini:")
print("=" * 100)
print(rag_context)

Full RAG Context Sent to Gemini:
[Source: rag.pdf, Chunk: 8, Score: 0.5670]
Like T5 [51] or BART, RAG can be ﬁne-tuned on any seq2seq task, whereby both the generator and retriever are jointly learned. There has been extensive previous work proposing architectures to enrich systems with non-parametric memory which are trained from scratch for speciﬁc tasks, e.g. memory networks [64, 55], stack- augmented networks [25] and memory layers [30]. In contrast, we explore a setting where both parametric and non-parametric memory components are pre-trained and pre-loaded with extensive knowledge. Crucially, by using pre-trained access mechanisms, the ability to access knowledge is present without additional training.

[Source: rag.pdf, Chunk: 42, Score: 0.5637]
This example shows how parametric and non-parametric memories work together—the non-parametric component helps to guide the generation, drawing out speciﬁc knowledge stored in the parametric memory. 4.4 Fact Veriﬁcation Table 2 shows ou

In [241]:
def display_gemini_rag_result(question, top_k=3):
    """
    Displays a full Gemini RAG result clearly:
    question, generated answer, and retrieved source chunks.
    """

    answer, sources, context = generate_rag_answer_gemini_embedding_only(
        question=question,
        top_k=top_k
    )

    print("Question:")
    print(question)

    print("\nGenerated Answer:")
    print("=" * 100)
    print(answer)

    print("\nRetrieved Source Chunks:")
    print("=" * 100)

    for _, row in sources.iterrows():
        print("File Name:", row["File Name"])
        print("Chunk Number:", row["Chunk Number"])
        print("Similarity Score:", round(row["Similarity Score"], 4))
        print("-" * 100)
        print(row["Chunk Text"][:1000])
        print("=" * 100)

In [242]:
display_gemini_rag_result(
    "What does BERT stand for?",
    top_k=3
)

Question:
What does BERT stand for?

Generated Answer:
The document context does not provide enough information.

Retrieved Source Chunks:
File Name: bert.pdf
Chunk Number: 16
Similarity Score: 0.4972
----------------------------------------------------------------------------------------------------
For ﬁne- tuning, the BERT model is ﬁrst initialized with the pre-trained parameters, and all of the param- eters are ﬁne-tuned using labeled data from the downstream tasks. Each downstream task has sep- arate ﬁne-tuned models, even though they are ini- tialized with the same pre-trained parameters. The question-answering example in Figure 1 will serve as a running example for this section. A distinctive feature of BERT is its uniﬁed ar- chitecture across different tasks. There is mini- mal difference between the pre-trained architec- ture and the ﬁnal downstream architecture.
File Name: bert.pdf
Chunk Number: 19
Similarity Score: 0.4921
-----------------------------------------------------

## 17.2 Observation from Embedding-Only RAG

The first RAG attempt used embedding-based retrieval only. For the question "What does BERT stand for?", the retrieved chunks were semantically related to BERT but did not contain the exact acronym definition.

Gemini correctly refused to answer because the required information was not present in the retrieved context.

This shows that the issue was not with the answer generation model. The issue was with retrieval. Embedding retrieval is useful for semantic and conceptual questions, but it may not always retrieve the best chunk for exact acronym or definition-based questions.

This motivates the use of hybrid retrieval, where TF-IDF and embedding-based retrieval are combined.

Now we will improve the RAG pipeline by combining:

**TF-IDF retrieval + Embedding retrieval**

This is better because:

- TF-IDF helps with exact keyword and acronym questions.
- Embeddings help with semantic and conceptual questions.

In [243]:
#Creating a Hybrid Retrieval Function
def retrieve_hybrid(query, top_k_tfidf=3, top_k_embedding=3):
    """
    Retrieves chunks using both TF-IDF and embedding-based retrieval.

    TF-IDF is useful for exact keyword, acronym, and definition-based questions.
    Embedding retrieval is useful for semantic and conceptual questions.

    Parameters:
    query (str): User question.
    top_k_tfidf (int): Number of TF-IDF chunks to retrieve.
    top_k_embedding (int): Number of embedding chunks to retrieve.

    Returns:
    pandas.DataFrame: Combined retrieved chunks without duplicates.
    """

    # Retrieve using TF-IDF
    tfidf_results = retrieve_tfidf(query, top_k=top_k_tfidf).copy()
    tfidf_results["Retrieval Method"] = "TF-IDF"

    # Retrieve using embeddings
    embedding_results = retrieve_embeddings(query, top_k=top_k_embedding).copy()
    embedding_results["Retrieval Method"] = "Embedding"

    # Combine both results
    combined_results = pd.concat(
        [tfidf_results, embedding_results],
        ignore_index=True
    )

    # Remove duplicate chunks
    combined_results = combined_results.drop_duplicates(
        subset=["Chunk ID"],
        keep="first"
    )

    # Reset index
    combined_results = combined_results.reset_index(drop=True)

    return combined_results

In [244]:
hybrid_test_results = retrieve_hybrid(
    "What does BERT stand for?",
    top_k_tfidf=3,
    top_k_embedding=3
)

display(hybrid_test_results[
    [
        "Chunk ID",
        "File Name",
        "Chunk Number",
        "Retrieval Method",
        "Similarity Score",
        "Chunk Text"
    ]
])

,Chunk ID,File Name,Chunk Number,Retrieval Method,Similarity Score,Chunk Text
0,bert.pdf_chunk_1,bert.pdf,1,TF-IDF,0.181605,Abstract We introduce a new language represent...
1,bert.pdf_chunk_7,bert.pdf,7,TF-IDF,0.096050,"(2018a), which uses a shallow concatenation of..."
2,bert.pdf_chunk_15,bert.pdf,15,TF-IDF,0.085000,Computer vision research has also demon- strat...
3,bert.pdf_chunk_16,bert.pdf,16,Embedding,0.497222,"For ﬁne- tuning, the BERT model is ﬁrst initia..."
4,bert.pdf_chunk_19,bert.pdf,19,Embedding,0.492141,A “sequence” refers to the in- put token seque...
5,bert.pdf_chunk_22,bert.pdf,22,Embedding,0.468686,"(2018a) and Radford et al. (2018), we do not u..."


In [245]:
#Hybrid RAG with Gemini
def generate_rag_answer_gemini(question, top_k_tfidf=3, top_k_embedding=3):
    """
    Generates an answer using Gemini API with hybrid retrieved document chunks.

    Steps:
    1. Retrieve relevant chunks using TF-IDF.
    2. Retrieve relevant chunks using embedding retrieval.
    3. Combine retrieved chunks and remove duplicates.
    4. Build context from retrieved chunks.
    5. Send context and question to Gemini.
    6. Return generated answer, retrieved source chunks, and context.
    """

    # Retrieve chunks using hybrid retrieval
    retrieved_chunks = retrieve_hybrid(
        query=question,
        top_k_tfidf=top_k_tfidf,
        top_k_embedding=top_k_embedding
    )

    # Build context from retrieved chunks
    context = build_rag_context(
        retrieved_chunks,
        max_chars_per_chunk=1000
    )

    # Create grounded prompt
    prompt = f"""
You are an academic document question-answering assistant.

Answer the question using only the provided context.
Do not use outside knowledge.

If the answer is not available in the context, say:
"The document context does not provide enough information."

Keep the answer clear, concise, and academic.

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate answer using Gemini
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    answer = response.text

    return answer, retrieved_chunks, context

In [246]:
def display_gemini_rag_result(question, top_k_tfidf=3, top_k_embedding=3):
    """
    Displays a full hybrid Gemini RAG result:
    question, generated answer, and retrieved source chunks.
    """

    answer, sources, context = generate_rag_answer_gemini(
        question=question,
        top_k_tfidf=top_k_tfidf,
        top_k_embedding=top_k_embedding
    )

    print("Question:")
    print(question)

    print("\nGenerated Answer:")
    print("=" * 100)
    print(answer)

    print("\nRetrieved Source Chunks:")
    print("=" * 100)

    for _, row in sources.iterrows():
        print("File Name:", row["File Name"])
        print("Chunk Number:", row["Chunk Number"])
        print("Retrieval Method:", row["Retrieval Method"])
        print("Similarity Score:", round(row["Similarity Score"], 4))
        print("-" * 100)
        print(row["Chunk Text"][:1000])
        print("=" * 100)

In [247]:
display_gemini_rag_result(
    "What does BERT stand for?",
    top_k_tfidf=3,
    top_k_embedding=3
)

Question:
What does BERT stand for?

Generated Answer:
BERT stands for Bidirectional Encoder Representations from Transformers.

Retrieved Source Chunks:
File Name: bert.pdf
Chunk Number: 1
Retrieval Method: TF-IDF
Similarity Score: 0.1816
----------------------------------------------------------------------------------------------------
Abstract We introduce a new language representa- tion model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language repre- sentation models (Peters et al., 2018a; Rad- ford et al., 2018), BERT is designed to pre- train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a re- sult, the pre-trained BERT model can be ﬁne- tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language inference, without substantial task- speciﬁc architecture modi

In [248]:
rag_test_questions = [
    "What is the Transformer model based on?",
    "What does BERT stand for?",
    "What is Retrieval-Augmented Generation?",
    "Why is attention useful in sequence transduction?",
    "How does RAG combine parametric memory and non-parametric memory?"
]

rag_results = []

for question in rag_test_questions:
    answer, sources, context = generate_rag_answer_gemini(
        question=question,
        top_k_tfidf=3,
        top_k_embedding=3
    )

    source_list = []

    for _, row in sources.iterrows():
        source_list.append(
            f"{row['File Name']} - Chunk {row['Chunk Number']} "
            f"[{row['Retrieval Method']}, Score: {row['Similarity Score']:.4f}]"
        )

    rag_results.append({
        "Question": question,
        "Generated Answer": answer,
        "Retrieved Sources": " | ".join(source_list)
    })

rag_results_df = pd.DataFrame(rag_results)

display(rag_results_df)

,Question,Generated Answer,Retrieved Sources
0,What is the Transformer model based on?,The Transformer model is based entirely on att...,attention_is_all_you_need.pdf - Chunk 51 [TF-I...
1,What does BERT stand for?,BERT stands for Bidirectional Encoder Represen...,"bert.pdf - Chunk 1 [TF-IDF, Score: 0.1816] | b..."
2,What is Retrieval-Augmented Generation?,Retrieval-Augmented Generation (RAG) is a gene...,"rag.pdf - Chunk 6 [TF-IDF, Score: 0.2136] | ra..."
3,Why is attention useful in sequence transduction?,Attention mechanisms are useful in sequence tr...,attention_is_all_you_need.pdf - Chunk 9 [TF-ID...
4,How does RAG combine parametric memory and non...,RAG models combine a pre-trained seq2seq trans...,"rag.pdf - Chunk 6 [TF-IDF, Score: 0.6846] | ra..."


In [249]:
rag_results_path = "outputs/evaluation/hybrid_gemini_rag_results.csv"

rag_results_df.to_csv(rag_results_path, index=False)

print("Hybrid Gemini RAG results saved successfully at:", rag_results_path)

Hybrid Gemini RAG results saved successfully at: outputs/evaluation/hybrid_gemini_rag_results.csv


In [250]:
comparison_question = "What does BERT stand for?"

embedding_answer, embedding_sources, embedding_context = generate_rag_answer_gemini_embedding_only(
    comparison_question,
    top_k=3
)

hybrid_answer, hybrid_sources, hybrid_context = generate_rag_answer_gemini(
    comparison_question,
    top_k_tfidf=3,
    top_k_embedding=3
)

print("Question:")
print(comparison_question)

print("\nEmbedding-Only RAG Answer:")
print("=" * 100)
print(embedding_answer)

print("\nHybrid RAG Answer:")
print("=" * 100)
print(hybrid_answer)

Question:
What does BERT stand for?

Embedding-Only RAG Answer:
The document context does not provide enough information.

Hybrid RAG Answer:
BERT stands for Bidirectional Encoder Representations from Transformers.


## 17.3 Hybrid Retrieval Discussion

The embedding-only RAG pipeline showed an important limitation. For the question "What does BERT stand for?", embedding retrieval returned chunks that were semantically related to BERT, but those chunks did not contain the exact acronym definition. Gemini correctly refused to answer because the answer was not available in the retrieved context.

To improve this, hybrid retrieval was introduced. Hybrid retrieval combines TF-IDF retrieval and embedding-based retrieval.

TF-IDF is strong for exact keyword, acronym, and definition-based questions because it relies on word overlap. Embedding retrieval is strong for conceptual questions because it captures semantic similarity.

By combining both methods, the final RAG system receives a stronger and more diverse set of context chunks before answer generation. This improves the chance that the answer is grounded in the retrieved academic document content.

## 17.4 RAG Discussion

The final RAG-based question-answering system combines retrieval and generation.

The retrieval component first identifies relevant chunks from the academic document collection. The hybrid retriever combines TF-IDF and embedding-based retrieval, allowing the system to benefit from both exact keyword matching and semantic similarity.

The retrieved chunks are then passed to Gemini as context. Gemini is instructed to answer only from the provided context and not from outside knowledge.

This makes the system more grounded than directly asking a language model a question without document context. It is also more explainable because the retrieved source chunks are displayed along with the generated answer.

Compared with retrieval alone, RAG produces a direct natural-language answer. Compared with a language model alone, RAG provides document-grounded context and source transparency.

## 17.5 RAG Limitations

Although RAG improves document question answering, it has some limitations.

First, the generated answer depends on the retrieved context. If the retrieval system returns weak or unrelated chunks, the answer may also be weak.

Second, RAG does not guarantee perfect factual accuracy. The answer must still be checked against the retrieved source chunks.

Third, this project uses a small document collection and simple sentence-based chunking. Larger document collections may require more advanced indexing, reranking, hybrid scoring, or improved chunking strategies.

Finally, Gemini API requires an API key and internet access. If an API key is not available, a local model such as FLAN-T5 can be used as an alternative, although answer quality may be weaker.

## 18. Evaluation and Comparison

This phase evaluates the document question-answering system using a small set of sample questions.

The goal is not to perform large-scale benchmark evaluation, but to qualitatively compare how different retrieval and answer generation methods behave.

The evaluation compares:

1. **TF-IDF Retrieval**
   - A classical keyword-based retrieval method.
   - Strong when the question shares exact terms with the document.

2. **Embedding-Based Retrieval**
   - A semantic retrieval method.
   - Strong when the question is conceptually similar to the document even if wording differs.

3. **Hybrid RAG**
   - Combines TF-IDF and embedding retrieval.
   - Uses Gemini API to generate a final answer from retrieved chunks.
   - Displays source chunks for explainability.

The evaluation focuses on relevance, groundedness, and answer quality.

In [251]:
# Define evaluation questions for the QA system

evaluation_questions = [
    {
        "Question": "What is the Transformer model based on?",
        "Expected Idea": "The Transformer is based on attention mechanisms, especially self-attention."
    },
    {
        "Question": "What does BERT stand for?",
        "Expected Idea": "BERT stands for Bidirectional Encoder Representations from Transformers."
    },
    {
        "Question": "What is Retrieval-Augmented Generation?",
        "Expected Idea": "RAG combines a generative model with retrieved external knowledge or non-parametric memory."
    },
    {
        "Question": "Why is attention useful in sequence transduction?",
        "Expected Idea": "Attention helps model relationships between positions in a sequence and supports parallelization."
    },
    {
        "Question": "How does RAG combine parametric memory and non-parametric memory?",
        "Expected Idea": "RAG uses a pre-trained seq2seq model as parametric memory and a dense vector index as non-parametric memory."
    }
]

evaluation_questions_df = pd.DataFrame(evaluation_questions)

display(evaluation_questions_df)

,Question,Expected Idea
0,What is the Transformer model based on?,The Transformer is based on attention mechanis...
1,What does BERT stand for?,BERT stands for Bidirectional Encoder Represen...
2,What is Retrieval-Augmented Generation?,RAG combines a generative model with retrieved...
3,Why is attention useful in sequence transduction?,Attention helps model relationships between po...
4,How does RAG combine parametric memory and non...,RAG uses a pre-trained seq2seq model as parame...


In [252]:
# Compare TF-IDF and embedding-based retrieval for each evaluation question

retrieval_comparison_rows = []

for item in evaluation_questions:
    question = item["Question"]

    tfidf_top = retrieve_tfidf(question, top_k=1).iloc[0]
    embedding_top = retrieve_embeddings(question, top_k=1).iloc[0]

    retrieval_comparison_rows.append({
        "Question": question,
        "Expected Idea": item["Expected Idea"],

        "TF-IDF Top File": tfidf_top["File Name"],
        "TF-IDF Top Chunk": tfidf_top["Chunk Number"],
        "TF-IDF Score": round(tfidf_top["Similarity Score"], 4),
        "TF-IDF Preview": tfidf_top["Chunk Text"][:300],

        "Embedding Top File": embedding_top["File Name"],
        "Embedding Top Chunk": embedding_top["Chunk Number"],
        "Embedding Score": round(embedding_top["Similarity Score"], 4),
        "Embedding Preview": embedding_top["Chunk Text"][:300]
    })

retrieval_comparison_df = pd.DataFrame(retrieval_comparison_rows)

display(retrieval_comparison_df)

,Question,Expected Idea,TF-IDF Top File,TF-IDF Top Chunk,TF-IDF Score,TF-IDF Preview,Embedding Top File,Embedding Top Chunk,Embedding Score,Embedding Preview
0,What is the Transformer model based on?,The Transformer is based on attention mechanis...,attention_is_all_you_need.pdf,51,0.3011,"Conclusion In this work, we presented the Tran...",attention_is_all_you_need.pdf,11,0.5058,At each step the model is auto-regressive [10]...
1,What does BERT stand for?,BERT stands for Bidirectional Encoder Represen...,bert.pdf,1,0.1816,Abstract We introduce a new language represent...,bert.pdf,16,0.4972,"For ﬁne- tuning, the BERT model is ﬁrst initia..."
2,What is Retrieval-Augmented Generation?,RAG combines a generative model with retrieved...,rag.pdf,6,0.2136,"Here, we bring hybrid parametric and non-param...",rag.pdf,60,0.5833,"This said, RAG techniques may work well in the..."
3,Why is attention useful in sequence transduction?,Attention helps model relationships between po...,attention_is_all_you_need.pdf,9,0.3638,"Self-attention, sometimes called intra-attenti...",attention_is_all_you_need.pdf,6,0.6253,Recurrent models typically factor computation ...
4,How does RAG combine parametric memory and non...,RAG uses a pre-trained seq2seq model as parame...,rag.pdf,6,0.6846,"Here, we bring hybrid parametric and non-param...",rag.pdf,8,0.5670,"Like T5 [51] or BART, RAG can be ﬁne-tuned on ..."


In [253]:
# Generate Hybrid RAG answers for all evaluation questions

rag_evaluation_rows = []

for item in evaluation_questions:
    question = item["Question"]

    answer, sources, context = generate_rag_answer_gemini(
        question=question,
        top_k_tfidf=3,
        top_k_embedding=3
    )

    source_list = []

    for _, row in sources.iterrows():
        source_list.append(
            f"{row['File Name']} - Chunk {row['Chunk Number']} "
            f"[{row['Retrieval Method']}, Score: {row['Similarity Score']:.4f}]"
        )

    rag_evaluation_rows.append({
        "Question": question,
        "Expected Idea": item["Expected Idea"],
        "Hybrid RAG Answer": answer,
        "Retrieved Sources": " | ".join(source_list)
    })

rag_evaluation_df = pd.DataFrame(rag_evaluation_rows)

display(rag_evaluation_df)

,Question,Expected Idea,Hybrid RAG Answer,Retrieved Sources
0,What is the Transformer model based on?,The Transformer is based on attention mechanis...,The Transformer is a sequence transduction mod...,attention_is_all_you_need.pdf - Chunk 51 [TF-I...
1,What does BERT stand for?,BERT stands for Bidirectional Encoder Represen...,BERT stands for Bidirectional Encoder Represen...,"bert.pdf - Chunk 1 [TF-IDF, Score: 0.1816] | b..."
2,What is Retrieval-Augmented Generation?,RAG combines a generative model with retrieved...,Retrieval-Augmented Generation (RAG) is a gene...,"rag.pdf - Chunk 6 [TF-IDF, Score: 0.2136] | ra..."
3,Why is attention useful in sequence transduction?,Attention helps model relationships between po...,Attention mechanisms are useful in sequence tr...,attention_is_all_you_need.pdf - Chunk 9 [TF-ID...
4,How does RAG combine parametric memory and non...,RAG uses a pre-trained seq2seq model as parame...,"RAG combines parametric memory, which is a pre...","rag.pdf - Chunk 6 [TF-IDF, Score: 0.6846] | ra..."


In [254]:
# Display one full evaluation example clearly

evaluation_question = "What does BERT stand for?"

print("Question:")
print(evaluation_question)

print("\nTF-IDF Retrieval:")
print("=" * 100)
tfidf_result = retrieve_tfidf(evaluation_question, top_k=1).iloc[0]
print("File:", tfidf_result["File Name"])
print("Chunk:", tfidf_result["Chunk Number"])
print("Score:", round(tfidf_result["Similarity Score"], 4))
print(tfidf_result["Chunk Text"][:1000])

print("\nEmbedding Retrieval:")
print("=" * 100)
embedding_result = retrieve_embeddings(evaluation_question, top_k=1).iloc[0]
print("File:", embedding_result["File Name"])
print("Chunk:", embedding_result["Chunk Number"])
print("Score:", round(embedding_result["Similarity Score"], 4))
print(embedding_result["Chunk Text"][:1000])

print("\nHybrid RAG Answer:")
print("=" * 100)
answer, sources, context = generate_rag_answer_gemini(
    evaluation_question,
    top_k_tfidf=3,
    top_k_embedding=3
)
print(answer)

print("\nHybrid RAG Sources:")
print("=" * 100)

for _, row in sources.iterrows():
    print("File:", row["File Name"])
    print("Chunk:", row["Chunk Number"])
    print("Method:", row["Retrieval Method"])
    print("Score:", round(row["Similarity Score"], 4))
    print("-" * 100)
    print(row["Chunk Text"][:700])
    print("=" * 100)

Question:
What does BERT stand for?

TF-IDF Retrieval:
File: bert.pdf
Chunk: 1
Score: 0.1816
Abstract We introduce a new language representa- tion model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language repre- sentation models (Peters et al., 2018a; Rad- ford et al., 2018), BERT is designed to pre- train deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers. As a re- sult, the pre-trained BERT model can be ﬁne- tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language inference, without substantial task- speciﬁc architecture modiﬁcations. BERT is conceptually simple and empirically powerful. It obtains new state-of-the-art re- sults on eleven natural language processing tasks, including pushing the GLUE score to 80.5% (7.7% point absolute improvement), MultiNLI accuracy to

## 18.1 Qualitative Evaluation Criteria

The system is evaluated qualitatively using three simple criteria:

1. **Retrieval Relevance**
   - Did the retrieved chunks contain information related to the question?

2. **Answer Groundedness**
   - Was the generated answer supported by the retrieved chunks?

3. **Answer Quality**
   - Was the answer clear, concise, and useful?

The evaluation is qualitative because this project uses a small academic document collection and does not include a labelled benchmark dataset.

In [256]:
# Qualitative evaluation table

manual_evaluation = pd.DataFrame([
    {
        "Question": "What is the Transformer model based on?",
        "Best Performing Method": "Hybrid RAG",
        "Retrieval Relevance": "High",
        "Answer Groundedness": "High",
        "Answer Quality": "Good",
        "Comment": "The retrieved chunks mention attention and self-attention, allowing RAG to generate a grounded answer."
    },
    {
        "Question": "What does BERT stand for?",
        "Best Performing Method": "TF-IDF / Hybrid RAG",
        "Retrieval Relevance": "High after hybrid retrieval",
        "Answer Groundedness": "High",
        "Answer Quality": "Good",
        "Comment": "TF-IDF retrieved the exact definition better than embedding-only retrieval. Hybrid RAG improved the final answer."
    },
    {
        "Question": "What is Retrieval-Augmented Generation?",
        "Best Performing Method": "Hybrid RAG",
        "Retrieval Relevance": "High",
        "Answer Groundedness": "High",
        "Answer Quality": "Good",
        "Comment": "The retrieved RAG chunks explained parametric and non-parametric memory, supporting a clear answer."
    },
    {
        "Question": "Why is attention useful in sequence transduction?",
        "Best Performing Method": "Embedding Retrieval / Hybrid RAG",
        "Retrieval Relevance": "High",
        "Answer Groundedness": "High",
        "Answer Quality": "Good",
        "Comment": "Embedding retrieval performed well because this was a conceptual question."
    },
    {
        "Question": "How does RAG combine parametric memory and non-parametric memory?",
        "Best Performing Method": "TF-IDF / Hybrid RAG",
        "Retrieval Relevance": "High",
        "Answer Groundedness": "High",
        "Answer Quality": "Good",
        "Comment": "TF-IDF performed strongly because the question used exact technical terms from the document."
    }
])

display(manual_evaluation)

,Question,Best Performing Method,Retrieval Relevance,Answer Groundedness,Answer Quality,Comment
0,What is the Transformer model based on?,Hybrid RAG,High,High,Good,The retrieved chunks mention attention and sel...
1,What does BERT stand for?,TF-IDF / Hybrid RAG,High after hybrid retrieval,High,Good,TF-IDF retrieved the exact definition better t...
2,What is Retrieval-Augmented Generation?,Hybrid RAG,High,High,Good,The retrieved RAG chunks explained parametric ...
3,Why is attention useful in sequence transduction?,Embedding Retrieval / Hybrid RAG,High,High,Good,Embedding retrieval performed well because thi...
4,How does RAG combine parametric memory and non...,TF-IDF / Hybrid RAG,High,High,Good,TF-IDF performed strongly because the question...


In [257]:
# Save evaluation outputs

retrieval_comparison_path = "outputs/evaluation/retrieval_comparison.csv"
rag_evaluation_path = "outputs/evaluation/rag_evaluation.csv"
manual_evaluation_path = "outputs/evaluation/manual_qualitative_evaluation.csv"

retrieval_comparison_df.to_csv(retrieval_comparison_path, index=False)
rag_evaluation_df.to_csv(rag_evaluation_path, index=False)
manual_evaluation.to_csv(manual_evaluation_path, index=False)

print("Evaluation files saved successfully:")
print(retrieval_comparison_path)
print(rag_evaluation_path)
print(manual_evaluation_path)

Evaluation files saved successfully:
outputs/evaluation/retrieval_comparison.csv
outputs/evaluation/rag_evaluation.csv
outputs/evaluation/manual_qualitative_evaluation.csv


## 18.2 Evaluation Discussion

The evaluation shows that different retrieval methods have different strengths.

TF-IDF performed well for questions that used exact technical terms from the documents. For example, it retrieved strong results for questions such as "What does BERT stand for?" and "How does RAG combine parametric memory and non-parametric memory?" because these questions contain keywords that appear directly in the source documents.

Embedding-based retrieval performed better for broader conceptual questions. For example, questions about why attention is useful in sequence transduction were well supported by embedding retrieval because semantic similarity helped retrieve conceptually relevant chunks.

The embedding-only RAG experiment also showed an important limitation. For the BERT acronym question, embedding retrieval returned chunks that were related to BERT but did not contain the exact definition. Gemini correctly refused to answer because the necessary information was not available in the retrieved context.

To address this, hybrid retrieval was introduced. By combining TF-IDF and embedding retrieval, the final RAG system benefits from both exact keyword matching and semantic similarity. This improved the quality of the retrieved context and made the generated answers more reliable.

Overall, the hybrid RAG approach provided the strongest final system because it retrieved diverse and relevant context before generating answers.

## 18.3 Evaluation Limitations

This evaluation is qualitative and based on a small set of sample questions. It is suitable for a course project because the goal is to demonstrate the full NLP pipeline and compare retrieval methods, rather than to achieve benchmark-level performance.

The evaluation has some limitations:

1. The document collection is small.
2. Only five sample questions were used.
3. The relevance and groundedness judgements are manually assessed.
4. The retrieved chunks depend on chunk size and overlap.
5. Gemini API output may vary slightly across runs.
6. A larger evaluation would require labelled question-answer pairs and quantitative metrics.

Future improvements could include a larger test set, human evaluation, automatic metrics, reranking, hybrid scoring, and more advanced chunking strategies.

## 19. Streamlit Application for GitHub Deployment

After completing the main NLP and RAG pipeline in Google Colab, a Streamlit application will be created as a separate GitHub deployment file.

The purpose of the Streamlit app is to provide an interactive interface where users can upload academic or technical PDF documents, ask questions, and receive document-grounded answers. The app reuses the main project logic developed in this notebook, including PDF text extraction, text cleaning, document chunking, TF-IDF retrieval, embedding-based retrieval, hybrid retrieval, and Gemini-based RAG answer generation.

The Streamlit implementation is kept separate from the notebook to keep the notebook focused on demonstrating the NLP course concepts step by step.

The Streamlit app will be stored in the following location:

`app/streamlit_app.py`

The app will support:

1. Uploading academic or technical PDFs
2. Extracting machine-readable text
3. Cleaning and chunking uploaded documents
4. Building TF-IDF and FAISS indexes
5. Asking questions over uploaded documents
6. Generating grounded answers using Gemini API
7. Displaying retrieved source chunks for explainability

## 20. Results and Discussion

This project successfully developed **DocuMind RAG**, an explainable NLP and Retrieval-Augmented Question Answering system for academic and technical documents.

The system was developed step by step, beginning with classical NLP preprocessing and progressing toward retrieval-based and generation-based question answering. The project demonstrates that a strong document QA system should not jump directly to large language models, but should first include careful text extraction, preprocessing, representation, retrieval, and evaluation.

The final system supports question answering over academic and technical documents by combining:

1. Classical NLP preprocessing
2. Bag of Words and TF-IDF encoding
3. TF-IDF baseline retrieval
4. Sentence embedding retrieval with FAISS
5. Hybrid retrieval
6. Gemini-based RAG answer generation
7. Source chunk display for explainability

The final system is explainable because it does not only generate an answer. It also displays the retrieved source chunks used to support the answer.

## 20.1 Summary of Completed NLP Pipeline

The project began with machine-readable text extraction from selected academic research papers. The extracted text was cleaned to remove major PDF-related noise such as page markers, email addresses, URLs, excessive whitespace, boilerplate text, and reference sections.

After cleaning, several classical NLP preprocessing steps were applied and demonstrated:

- Tokenization was used to split text into sentences and words.
- Normalization was used to standardise text by lowercasing and removing unnecessary punctuation while preserving important decimal values.
- Stopword removal was demonstrated to reduce common low-information words.
- Stemming and lemmatization were compared, with lemmatization selected as the preferred method because it preserves meaningful dictionary forms.
- POS tagging was used to identify grammatical categories such as nouns, verbs, adjectives, and adverbs.
- Dependency parsing was used to show grammatical relationships between words in sentences.

These steps helped demonstrate the complete NLP pipeline before moving into retrieval and RAG.

## 20.1 Summary of Completed NLP Pipeline

- **Text Cleaning**  
  Raw text extracted from the academic PDFs was cleaned to remove major PDF-related noise such as page markers, email addresses, URLs, excessive whitespace, boilerplate text, and reference sections. This helped make the text more readable and suitable for later NLP processing.

- **Tokenization**  
  The cleaned text was split into smaller units using sentence tokenization and word tokenization. Sentence tokenization helped divide long academic documents into meaningful sentence units, while word tokenization supported later preprocessing steps such as stopword removal, lemmatization, and encoding.

- **Normalization**  
  The text was normalised by converting it to lowercase, removing unnecessary punctuation and special characters, and standardising spacing. Decimal values such as `28.4`, `41.8`, and `3.5` were preserved because numerical values are important in academic and technical documents.

- **Stopword Removal**  
  Common English stopwords such as “the”, “is”, “and”, and “of” were removed from the normalised text. This helped reduce low-information words and made important technical terms more visible for classical NLP methods.

- **Stemming / Lemmatization**  
  Stemming and lemmatization were compared to show how words can be reduced to their base forms. Stemming was demonstrated for learning purposes, but spaCy-based lemmatization was used for document processing because it preserves meaningful dictionary forms such as `generating → generate`, `retrieving → retrieve`, and `models → model`.

- **POS Tagging**  
  Part-of-Speech tagging was applied to identify grammatical categories such as nouns, verbs, adjectives, and adverbs. This helped show how academic and technical text can be analysed linguistically before applying retrieval or machine learning methods.

- **Parsing**  
  Dependency parsing was demonstrated to identify grammatical relationships between words in a sentence. While POS tagging shows the role of each word, parsing shows how words are connected, such as subject, object, modifier, and root verb relationships.

- **Encoding Techniques**  
  Text was converted into numerical representations using Bag of Words, TF-IDF, and sentence embeddings. Bag of Words represented text through word counts, TF-IDF represented text using weighted term importance, and sentence embeddings represented text as dense semantic vectors. These encoding techniques connected the classical NLP pipeline to the later retrieval and RAG-based question-answering system.

## 20.2 Text Cleaning and Preprocessing Results

The text cleaning process improved the quality of the extracted PDF text. Raw PDF extraction often produced noise such as author emails, page markers, line breaks, formatting issues, and reference text. After cleaning, the documents became more suitable for NLP processing and retrieval.

One important observation was that cleaning directly affected retrieval quality. In earlier TF-IDF retrieval tests, reference-related chunks appeared as top results for some questions. After improving the cleaning function to remove reference sections more reliably, the retrieved chunks became more relevant and focused on the main academic content.

This shows that preprocessing is not just a theoretical NLP step. It has a direct impact on the performance of retrieval and question answering systems.

## 20.3 Encoding Results

Three encoding techniques were demonstrated: Bag of Words, TF-IDF, and sentence embeddings.

Bag of Words represented text using simple word-count vectors. This method was easy to understand and interpret, but it ignored word order and meaning.

TF-IDF improved on Bag of Words by assigning higher importance to distinctive terms. It was useful for identifying important technical terms and became the baseline retrieval method in the project.

Sentence embeddings represented text as dense semantic vectors. Unlike Bag of Words and TF-IDF, embeddings captured meaning rather than only exact word overlap. This made them useful for semantic retrieval and RAG.

The encoding phase showed how text can be transformed into numerical representations that support retrieval and question answering.

## 20.4 TF-IDF Retrieval Results

The TF-IDF retrieval system served as the classical NLP baseline.

TF-IDF performed well when the user question shared exact keywords with the document chunks. For example, it retrieved strong results for questions such as:

- "What does BERT stand for?"
- "What is the Transformer model based on?"
- "How does RAG combine parametric memory and non-parametric memory?"

The BERT question was a strong example of TF-IDF retrieval working well. Because the question contained the exact term "BERT", TF-IDF was able to retrieve the abstract chunk that directly stated that BERT stands for Bidirectional Encoder Representations from Transformers.

However, TF-IDF has limitations. It depends heavily on word overlap and may not retrieve the best result when a question uses different wording from the document. It also does not understand deeper semantic meaning.

## 20.5 Embedding-Based Retrieval Results

The embedding-based retrieval system used sentence embeddings and FAISS to retrieve semantically relevant chunks.

Embedding retrieval performed well for broader conceptual questions. For example, it retrieved relevant chunks for questions about attention, sequence transduction, Transformer architecture, and RAG concepts.

Unlike TF-IDF, embedding retrieval can identify related content even when the exact wording differs. This makes it useful for academic and technical documents, where users may ask questions in different words from the original text.

However, embedding retrieval did not always perform best for exact acronym or definition-based questions. For example, for the question "What does BERT stand for?", embedding retrieval returned chunks related to BERT architecture and token processing, but did not initially retrieve the chunk containing the exact acronym definition.

This showed that semantic similarity does not always guarantee that the most directly useful chunk will be ranked first.

## 20.6 Hybrid Retrieval Results

Hybrid retrieval was introduced after observing the limitations of embedding-only retrieval.

The hybrid retrieval system combines:

1. TF-IDF retrieval
2. Embedding-based retrieval

This approach improved the final RAG system because it allowed the system to benefit from both exact keyword matching and semantic similarity.

The BERT question demonstrated the value of hybrid retrieval clearly. Embedding-only retrieval did not retrieve the chunk containing the exact definition of BERT, so Gemini correctly responded that the document context did not provide enough information. After hybrid retrieval was introduced, the TF-IDF component retrieved the correct abstract chunk, allowing the final RAG system to answer correctly.

This shows that hybrid retrieval is especially useful for academic and technical documents because such documents contain both exact technical terms and broader conceptual explanations.

## 20.7 RAG-Based Question Answering Results

The final RAG system used retrieved document chunks as context for Gemini-based answer generation.

Compared with retrieval alone, RAG improved the user experience by generating direct natural-language answers. Instead of only showing chunks, the system produced concise answers grounded in the retrieved document context.

The system was also designed to be explainable. For every generated answer, the retrieved source chunks were displayed. This allowed the answer to be checked against the document evidence.

A key strength of the RAG system was that it was instructed to answer only from the provided context. This was demonstrated when embedding-only RAG could not answer the BERT acronym question because the retrieved chunks did not contain the answer. Gemini correctly refused to answer instead of relying on outside knowledge.

After hybrid retrieval was added, the final RAG system became stronger because it received better context before generating answers.

## 20.8 Streamlit Application Result

A Streamlit application was planned as the GitHub interface for DocuMind RAG.

The purpose of the Streamlit app is to allow users to interact with the system outside the notebook. The app can support PDF upload, text extraction, cleaning, chunking, retrieval, Gemini-based answer generation, and retrieved source chunk display.

Keeping the Streamlit app separate from the notebook is useful because the notebook remains focused on explaining the NLP course pipeline, while the app serves as a practical deployment interface for GitHub and portfolio presentation.

This separation makes the project stronger in two ways:

1. The notebook demonstrates academic understanding of NLP concepts.
2. The Streamlit app demonstrates practical implementation and usability.

## 20.9 Why Classical NLP Still Matters

A key finding of this project is that classical NLP techniques remain important even when modern LLMs and RAG systems are used.

Text cleaning had a direct effect on retrieval quality. When noisy reference sections were present, retrieval results were less accurate. After cleaning was improved, retrieval results became more relevant.

TF-IDF also remained useful. It performed strongly for exact keyword, acronym, and definition-based questions. This showed that classical keyword-based retrieval can sometimes outperform embedding retrieval for specific types of questions.

Tokenization, normalization, stopword removal, lemmatization, POS tagging, parsing, and encoding all helped demonstrate how raw text is gradually transformed into usable representations.

Therefore, the project shows that modern RAG systems are stronger when they are built on top of a clear and well-designed NLP pipeline.

## 20.10 Limitations

Although DocuMind RAG successfully demonstrates an explainable NLP and RAG-based document question-answering system, it has some limitations.

First, the document collection is small and contains only a few academic papers. A larger collection would better test scalability and retrieval performance.

Second, the chunking method is simple. The project uses sentence-based chunks with overlap, but more advanced methods such as section-based chunking or semantic chunking may improve retrieval quality.

Third, the evaluation is qualitative and based on a small number of manually selected questions. A larger evaluation set with ground-truth answers would allow more rigorous testing.

Fourth, the embedding model used in this project is lightweight. Larger or domain-specific embedding models may improve semantic retrieval for academic and technical documents.

Fifth, the RAG answer generation step uses Gemini API, which requires an API key and internet access.

Finally, the project focuses only on machine-readable text. Scanned PDFs, images, diagrams, and OCR-based extraction are outside the scope of this project.

## 20.11 Future Improvements

Several improvements could be made in future versions of DocuMind RAG.

1. **Larger document collection**  
   The system could be tested on more academic papers, lecture notes, manuals, and technical reports.

2. **Improved chunking**  
   Future work could use heading-based chunking, section-aware chunking, or semantic chunking.

3. **Hybrid scoring**  
   The current hybrid retrieval approach combines TF-IDF and embedding results. Future work could create a weighted hybrid score to rank combined results more systematically.

4. **Reranking**  
   A reranker model could be added after retrieval to improve the ordering of retrieved chunks.

5. **Better evaluation**  
   Future versions could use labelled question-answer pairs, human evaluation, precision@k, recall@k, answer faithfulness, and groundedness metrics.

6. **Streamlit deployment**  
   The Streamlit app can be expanded into a complete deployment where users can upload academic PDFs and ask questions interactively.

7. **OCR support**  
   OCR could be added to support scanned PDFs and image-heavy academic documents.

8. **Citation-style answers**  
   The generated answer could include clearer source citations based on document name and chunk number.

In [263]:
# Final project results summary

final_results_summary = pd.DataFrame([
    {
        "Component": "Text Extraction",
        "Result": "Extracted machine-readable text from academic PDFs.",
        "Importance": "Converted documents into text for NLP processing."
    },
    {
        "Component": "Text Cleaning and Preprocessing",
        "Result": "Removed major PDF noise and demonstrated tokenization, normalization, stopword removal, and lemmatization.",
        "Importance": "Improved text quality and satisfied classical NLP requirements."
    },
    {
        "Component": "Linguistic Analysis",
        "Result": "Demonstrated POS tagging and dependency parsing.",
        "Importance": "Showed grammatical and structural analysis of academic text."
    },
    {
        "Component": "Encoding Techniques",
        "Result": "Demonstrated Bag of Words, TF-IDF, and sentence embeddings.",
        "Importance": "Converted text into numerical representations."
    },
    {
        "Component": "TF-IDF Retrieval",
        "Result": "Performed well for exact keyword and definition-based questions.",
        "Importance": "Served as the classical NLP retrieval baseline."
    },
    {
        "Component": "Embedding Retrieval",
        "Result": "Performed well for semantic and conceptual questions.",
        "Importance": "Enabled meaning-based search using FAISS."
    },
    {
        "Component": "Hybrid Retrieval",
        "Result": "Combined TF-IDF and embedding retrieval.",
        "Importance": "Improved retrieval quality for both exact and conceptual questions."
    },
    {
        "Component": "RAG Answer Generation",
        "Result": "Generated grounded answers using Gemini API and retrieved context.",
        "Importance": "Produced explainable document-based answers."
    },
    {
        "Component": "Streamlit Interface",
        "Result": "Planned as a separate GitHub deployment interface.",
        "Importance": "Improves usability and shows practicality"
    }
])

display(final_results_summary)

,Component,Result,Importance
0,Text Extraction,Extracted machine-readable text from academic ...,Converted documents into text for NLP processing.
1,Text Cleaning and Preprocessing,Removed major PDF noise and demonstrated token...,Improved text quality and satisfied classical ...
2,Linguistic Analysis,Demonstrated POS tagging and dependency parsing.,Showed grammatical and structural analysis of ...
3,Encoding Techniques,"Demonstrated Bag of Words, TF-IDF, and sentenc...",Converted text into numerical representations.
4,TF-IDF Retrieval,Performed well for exact keyword and definitio...,Served as the classical NLP retrieval baseline.
5,Embedding Retrieval,Performed well for semantic and conceptual que...,Enabled meaning-based search using FAISS.
6,Hybrid Retrieval,Combined TF-IDF and embedding retrieval.,Improved retrieval quality for both exact and ...
7,RAG Answer Generation,Generated grounded answers using Gemini API an...,Produced explainable document-based answers.
8,Streamlit Interface,Planned as a separate GitHub deployment interf...,Improves usability and shows practicality


In [264]:
# Save final results summary

final_summary_path = "outputs/evaluation/final_results_summary.csv"

final_results_summary.to_csv(final_summary_path, index=False)

print("Final results summary saved successfully at:", final_summary_path)

Final results summary saved successfully at: outputs/evaluation/final_results_summary.csv


## 21. Conclusion

This project developed **DocuMind RAG**, an explainable NLP and Retrieval-Augmented Question Answering system for academic and technical documents.

The project successfully demonstrated the complete NLP pipeline, beginning with text extraction from academic PDFs and progressing through text cleaning, tokenization, normalization, stopword removal, stemming and lemmatization, POS tagging, dependency parsing, and encoding techniques such as Bag of Words, TF-IDF, and sentence embeddings.

After completing the classical NLP pipeline, the project built and compared multiple retrieval approaches. TF-IDF retrieval was used as the classical baseline and performed well for exact keyword, acronym, and definition-based questions. Embedding-based retrieval using FAISS was used for semantic search and performed well for broader conceptual questions. Hybrid retrieval was then introduced to combine the strengths of both methods.

The final RAG system used Gemini API to generate answers from retrieved document chunks. This made the system more useful than retrieval alone because it produced direct natural-language answers. At the same time, the system remained explainable because retrieved source chunks were displayed with each answer.

Overall, the project shows that effective document question answering requires more than simply applying an LLM. A strong QA system depends on careful preprocessing, meaningful text representation, reliable retrieval, and grounded answer generation.